# D4_switching_validation — Kobayashi方式の正当性検証

資源計数（`resource_analysis.ipynb` 第I部）とは**独立に**、符号・論理ゲート・プロトコル・測定回路の正しさを確認する。
各検査は `record(name, passed, detail)` で記録し、最後に `results/validation_d4.json` に保存する（`resource_analysis.ipynb` はこれを読み、検証状況を表示する）。

| ID | 内容 | 方法 |
|---|---|---|
| V1 | 安定化子の定義が [KZH] と一致 | Fig. 4 から読み取った特徴（境界ごとの生成元の種類、CZ の位置）との照合 |
| V2 | 符号空間の整合性 | $C_Z$ 保存・$C_Z$ 上の可換性・$S^2=1$・$k=1$（$\mathbb F_2$ 厳密）＋負の対照 |
| V3 | 独立実装による照合 | $L=1$ を 埋め込みの D4 ライブラリを使わず密ベクトルで実装し、符号次元と論理 $T^\dagger$ を確認 |
| V4 | 論理演算子と距離 | MILP の距離を全探索で照合（小さい $L$） |
| V5 | Knill–Laflamme 条件 | $L=2$ の全 weight≤2 Pauli |
| V5b | D4 相の KL 距離 $=L$ | $L=3..5$（6）：低重み安定化子シフトの厳密列挙＋対角分解による厳密下界 |
| V6 | transversal $U$ | $USU^\dagger=S'$ の局所厳密列挙、論理位相、$V(C_Z)=C_Z$、規約を変えた負の対照 |
| V7 | プロトコル全体（厳密） | $L=1,2$ 疎状態ベクトル、ランダム入力・ランダム分岐、補正を外した負の対照、測定結果の統計 |
| V7b | 回路と位相公式の照合 | $L=3,4$：実際の測定・$U$・補正回路を配置入力に作用させ、同じ分岐での振幅比が $F$ と一致 |
| V8 | プロトコル全体（大きな $L$） | 配置ごとの位相 $F(a)$ が論理剰余類上で一定であることの局所厳密証明（$L\le 8$） |
| V9 | 測定ガジェット | 制御-$S$ 回路＋$X$ 測定 = 理想射影測定（密行列） |
| V10 | スケジュールの妥当性 | 同一クラス内の異なるガジェットのゲートが可換（小行列で独立に確認）、$U$ のゲートが互いに可換 |
| V11 | 資源計数の独立照合 | 幾何からの直接公式との一致 |

In [1]:
# ==== 環境設定（Google Colab / ローカル共通）====
# このノートブックは単独で実行できる（共通コードは下に埋め込まれており、他のファイルを読み込まない）。
# 出力（results/, figures/）はこのランタイムの作業ディレクトリに保存される。Colab ではランタイム終了で
# 消えるので、残したい場合は Google Drive を使う：
#   from google.colab import drive; drive.mount('/content/drive')
#   os.makedirs('/content/drive/MyDrive/D4_vs_MSD', exist_ok=True); os.chdir('/content/drive/MyDrive/D4_vs_MSD')
import os, sys, subprocess, json
IN_COLAB = 'google.colab' in sys.modules
RESULTS, FIGURES = 'results', 'figures'
os.makedirs(RESULTS, exist_ok=True); os.makedirs(FIGURES, exist_ok=True)
SEED = 20261024
print('cwd =', os.getcwd(), '| Colab =', IN_COLAB)

cwd = /tmp/claude-0/-home-user-2026-10-24-StudyWithClaude/fee9d03f-2fbf-502f-b670-8966176acf54/scratchpad/run3/d4 | Colab = False


In [2]:
# ==== 外部ライブラリ（Colab に無いものだけをインストール）====
import importlib.util
for _mod, _pkg in [('stim', 'stim'), ('scipy', 'scipy'), ('sympy', 'sympy'), ('pandas', 'pandas'),
                   ('matplotlib', 'matplotlib')]:
    if importlib.util.find_spec(_mod) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg], check=True)
import numpy as np, pandas as pd, scipy
print('numpy', np.__version__, '| scipy', scipy.__version__, '| pandas', pd.__version__)

numpy 2.5.3 | scipy 1.18.1 | pandas 3.0.5


## 共通ライブラリ（埋め込み・関数定義のみ）
Google Colab ではノートブックごとに別のランタイムで実行されるため、他のファイルを読み込まずに済むよう、共通コードをこのノートブックに埋め込んでいる（各ノートの共通コードは同一のソースから自動生成されており、内容は一致する）。
**このセクションは上から順に実行するだけでよく、編集する必要はない**（Colab では見出しの左の三角で折りたためる）。

### 共通ライブラリ（F2 線形代数・ゲート／ガジェット・スケジューラ・Clifford+T 分解・計数ルール R1–R3）

In [3]:
"""common_lib: F2 linear algebra, gate/gadget model, scheduler, decompositions, cost model rules."""
import itertools, math
import numpy as np
from collections import Counter, defaultdict

In [4]:
# ---------------------------------------------------------------- F2 algebra
def f2_rref(M):
    """Row-reduce a uint8 matrix over F2. Returns (R, pivot_cols)."""
    M = (np.array(M, dtype=np.uint8) & 1).copy()
    rows, cols = M.shape
    piv = []
    r = 0
    for c in range(cols):
        if r >= rows:
            break
        nz = np.nonzero(M[r:, c])[0]
        if len(nz) == 0:
            continue
        p = r + nz[0]
        if p != r:
            M[[r, p]] = M[[p, r]]
        others = np.nonzero(M[:, c])[0]
        others = others[others != r]
        if len(others):
            M[others] ^= M[r]
        piv.append(c)
        r += 1
    return M[:r], piv


def f2_rank(M):
    if len(M) == 0:
        return 0
    return len(f2_rref(M)[1])


def f2_nullspace(M, n=None):
    """Basis (rows) of {x : M x = 0} over F2."""
    M = np.array(M, dtype=np.uint8)
    if M.size == 0:
        return np.eye(n, dtype=np.uint8)
    n = M.shape[1]
    Rm, piv = f2_rref(M)
    free = [c for c in range(n) if c not in set(piv)]
    basis = []
    for f in free:
        v = np.zeros(n, dtype=np.uint8)
        v[f] = 1
        for i, p in enumerate(piv):
            if Rm[i, f]:
                v[p] = 1
        basis.append(v)
    return np.array(basis, dtype=np.uint8).reshape(len(basis), n)


def f2_in_rowspace(Rref, piv, v):
    """Is v in the row space of the RREF matrix Rref (pivots piv)?"""
    v = np.array(v, dtype=np.uint8) & 1
    w = v.copy()
    for i, p in enumerate(piv):
        if w[p]:
            w ^= Rref[i]
    return not w.any()


def f2_solve(A, b):
    """Solve x A = b (x row vector, A rows = generators). Returns x or None."""
    A = np.array(A, dtype=np.uint8)
    m, n = A.shape
    aug = np.concatenate([A, np.eye(m, dtype=np.uint8)], axis=1)
    Rm, piv = f2_rref(aug)
    w = np.concatenate([np.array(b, dtype=np.uint8) & 1, np.zeros(m, dtype=np.uint8)])
    for i, p in enumerate(piv):
        if p >= n:
            break
        if w[p]:
            w ^= Rm[i]
    if w[:n].any():
        return None
    return w[n:]




import itertools
from collections import Counter, defaultdict

ARITY = {'INIT_X': 1, 'INIT_Z': 1, 'MEAS_X': 1, 'MEAS_Z': 1, 'H': 1, 'X': 1, 'Z': 1, 'S': 1, 'SDG': 1,
         'T': 1, 'TDG': 1, 'CNOT': 2, 'CZ': 2, 'CS': 2, 'CSDG': 2, 'CCZ': 3}
NONCLIFFORD = {'T', 'TDG', 'CS', 'CSDG', 'CCZ'}

In [5]:
# ------------------------------------------------------------------ minimum-weight logical (MILP)
def min_weight_odd(Hc, lvec, cols):
    """min |x| over x in F2^cols with Hc x = 0 and lvec . x = 1 (exact MILP, HiGHS)."""
    from scipy.optimize import milp, LinearConstraint, Bounds
    H = np.array(Hc, dtype=float)[:, cols]
    H = H[np.any(H, axis=1)]
    l = np.array(lvec, dtype=float)[cols]
    m, n = H.shape
    # variables: x (n, binary), y (m, integer), z (1, integer)
    nv = n + m + 1
    c = np.concatenate([np.ones(n), np.zeros(m + 1)])
    A1 = np.concatenate([H, -2 * np.eye(m), np.zeros((m, 1))], axis=1)
    A2 = np.concatenate([l[None, :], np.zeros((1, m)), -2 * np.ones((1, 1))], axis=1)
    cons = [LinearConstraint(A1, 0, 0), LinearConstraint(A2, 1, 1)]
    integrality = np.ones(nv)
    ub = np.concatenate([np.ones(n), np.full(m + 1, np.inf)])
    res = milp(c, constraints=cons, integrality=integrality, bounds=Bounds(np.zeros(nv), ub))
    assert res.status == 0, res.message
    x = np.round(res.x[:n]).astype(np.uint8)
    out = np.zeros(len(lvec), dtype=np.uint8)
    out[cols] = x
    return int(x.sum()), out

In [6]:
# ------------------------------------------------------------------ gadgets
def pauli_check_gadget(anc, xq=(), zq=()):
    """Measure X(xq) Z(zq) with ancilla anc (standard CNOT circuit)."""
    g = [('INIT_X', anc)]
    g += [('CNOT', anc, q) for q in sorted(xq)]           # ancilla controls -> X on data
    g += [('CZ', anc, q) for q in sorted(zq)]             # controlled Z on data
    g += [('MEAS_X', anc)]
    return g


def z_check_gadget(anc, zq):
    g = [('INIT_Z', anc)] + [('CNOT', q, anc) for q in sorted(zq)] + [('MEAS_Z', anc)]
    return g


def clifford_x_gadget(anc, parts):
    """Measure a product of X-type Clifford operators S = S_1 S_2 ... with
    S_k = X(s_k) prod CZ(pairs_k): ancilla |+>, controlled-S (= CNOTs from anc and
    CCZ(anc,i,j) for each CZ pair; rightmost factor first), measure X.
    parts: list of (xqubits, czpairs) in operator order S_1 S_2 ..."""
    g = [('INIT_X', anc)]
    for xq, pairs in reversed(parts):
        g += [('CCZ', anc, i, j) for (i, j) in pairs]      # CZ part acts first within S_k
        g += [('CNOT', anc, q) for q in sorted(xq)]
    g += [('MEAS_X', anc)]
    return g

In [7]:
# ------------------------------------------------------------------ scheduling
def asap_layers(gates):
    """Greedy ASAP layering of a gate list respecting qubit order (no reordering of gates that
    share a qubit).  Returns list of layers (lists of gates)."""
    ready = defaultdict(int)
    layers = []
    for gt in gates:
        qs = gt[1:]
        t = max(ready[q] for q in qs)
        while len(layers) <= t:
            layers.append([])
        layers[t].append(gt)
        for q in qs:
            ready[q] = t + 1
    return layers


def gadget_support(gad):
    s = set()
    for gt in gad:
        s |= set(gt[1:])
    return s


def disjoint_groups(gadgets):
    """Greedy colouring: partition gadgets so that gadgets in a group touch disjoint qubits.
    Measurements within a group are then exactly simultaneous and the sequential-group
    schedule is valid for commuting (on the relevant subspace) generators."""
    groups = []          # list of (used_qubits, [gadgets])
    order = sorted(range(len(gadgets)), key=lambda i: -len(gadget_support(gadgets[i])))
    for i in order:
        sup = gadget_support(gadgets[i])
        for used, members in groups:
            if not (used & sup):
                used |= sup
                members.append(gadgets[i])
                break
        else:
            groups.append((set(sup), [gadgets[i]]))
    return [m for _, m in groups]


def layer_duration(layer, dur):
    return max(dur[g[0]] for g in layer)


def circuit_time(layers, dur):
    return sum(layer_duration(l, dur) for l in layers)


def round_schedule(gadgets, dur):
    """Upper-bound schedule of one round: disjoint groups executed sequentially, each group
    ASAP-layered.  Returns dict with depth (layers), time (weighted by dur), groups."""
    groups = disjoint_groups(gadgets)
    depth = 0
    time = 0
    for grp in groups:
        gl = asap_layers([gt for gad in grp for gt in gad])
        depth += len(gl)
        time += circuit_time(gl, dur)
    return {'groups': len(groups), 'depth_ub': depth, 'time_ub': time}


def round_lower_bound(gadgets, dur):
    """Lower bound on the duration of one round: every qubit executes its gates sequentially,
    so time >= max_q sum of durations of gates acting on q."""
    busy = defaultdict(float)
    cnt = defaultdict(int)
    for gad in gadgets:
        for gt in gad:
            for q in gt[1:]:
                busy[q] += dur[gt[0]]
                cnt[q] += 1
    return {'depth_lb': max(cnt.values()), 'time_lb': max(busy.values())}


def gate_counts(gates):
    c = Counter(g[0] for g in gates)
    return dict(c)

In [8]:
# ------------------------------------------------------------------ decompositions
# Explicit Clifford+T decompositions (verified numerically in the validation notebook).
def ccz_to_clifford_t(a, b, c):
    """CCZ with 7 T/T^dag and 6 CNOT (standard Toffoli decomposition, Nielsen & Chuang Fig. 4.9,
    with the target Hadamards removed)."""
    return [('CNOT', b, c), ('TDG', c), ('CNOT', a, c), ('T', c), ('CNOT', b, c), ('TDG', c),
            ('CNOT', a, c), ('T', b), ('T', c), ('CNOT', a, b), ('T', a), ('TDG', b), ('CNOT', a, b)]


def cs_to_clifford_t(a, b):
    """CS = diag(1,1,1,i) = T_a T_b CNOT(a,b) T^dag_b CNOT(a,b)."""
    return [('T', a), ('T', b), ('CNOT', a, b), ('TDG', b), ('CNOT', a, b)]


def csdg_to_clifford_t(a, b):
    return [('TDG', a), ('TDG', b), ('CNOT', a, b), ('T', b), ('CNOT', a, b)]


def decompose(gates):
    out = []
    for g in gates:
        if g[0] == 'CCZ':
            out += ccz_to_clifford_t(*g[1:])
        elif g[0] == 'CS':
            out += cs_to_clifford_t(*g[1:])
        elif g[0] == 'CSDG':
            out += csdg_to_clifford_t(*g[1:])
        else:
            out.append(g)
    return out


DUR_UNIT = {k: 1 for k in ARITY}            # every primitive = 1 time unit (ASSUMPTION, parametrised)

In [9]:
# ------------------------------------------------- commuting-class scheduler
DIAG_GATES = {'CZ', 'CCZ', 'CS', 'CSDG', 'T', 'TDG', 'S', 'SDG', 'Z'}


def gate_roles(gt):
    """Per-qubit role of an entangling/1q unitary: 'Z' (diagonal in Z basis), 'X' (X-type)."""
    name = gt[0]
    if name in DIAG_GATES:
        return {q: 'Z' for q in gt[1:]}
    if name == 'CNOT':
        return {gt[1]: 'Z', gt[2]: 'X'}
    if name == 'X':
        return {gt[1]: 'X'}
    return {q: '?' for q in gt[1:]}      # H, INIT, MEAS: never shared between gadgets


def gadget_roles(gad):
    roles = {}
    for gt in gad:
        for q, r in gate_roles(gt).items():
            if roles.get(q, r) != r:
                roles[q] = '?'
            else:
                roles[q] = r
    return roles


def commuting_classes(gadgets, n_orders=6, seed=0):
    """Best (fewest classes) of several greedy colourings (different gadget orders)."""
    import random
    best = None
    orders = [list(range(len(gadgets)))]
    orders.append(sorted(orders[0], key=lambda i: tuple(sorted(set(gadget_roles(gadgets[i]).values())))))
    rnd = random.Random(seed)
    for _ in range(n_orders):
        o = orders[0][:]
        rnd.shuffle(o)
        orders.append(o)
    for o in orders:
        cl = _commuting_classes_greedy([gadgets[i] for i in o])
        if best is None or len(cl) < len(best):
            best = cl
    return best


def _commuting_classes_greedy(gadgets):
    """Greedy colouring of the gadget conflict graph.  Two gadgets are compatible if on every
    shared qubit both act with the same role ('Z' or 'X'); then all their gates commute and the
    joint circuit is equivalent to measuring them one after another (sufficient condition)."""
    roles = [gadget_roles(g) for g in gadgets]
    classes = []     # list of (role_map, [idx])
    for i, rm in enumerate(roles):
        placed = False
        for cmap, members in classes:
            ok = all(cmap.get(q, r) == r and r != '?' for q, r in rm.items() if q in cmap)
            if ok:
                for q, r in rm.items():
                    cmap[q] = r
                members.append(i)
                placed = True
                break
        if not placed:
            classes.append((dict(rm), [i]))
    return [[gadgets[i] for i in m] for _, m in classes]


def class_layers(gadgets):
    """Layering of one commuting class: init layer, greedy layering of the (mutually commuting)
    entangling gates, measurement layer."""
    inits = [g for gad in gadgets for g in gad if g[0].startswith('INIT')]
    meas = [g for gad in gadgets for g in gad if g[0].startswith('MEAS')]
    body = [g for gad in gadgets for g in gad if not (g[0].startswith('INIT') or g[0].startswith('MEAS'))]
    # greedy: most-constrained gates first
    body = sorted(body, key=lambda g: -len(g))
    layers = []
    busy = []
    for g in body:
        qs = set(g[1:])
        for li, b in enumerate(busy):
            if not (b & qs) and layers[li][0][0] == g[0]:     # keep one gate type per layer
                b |= qs
                layers[li].append(g)
                break
        else:
            busy.append(set(qs))
            layers.append([g])
    return ([inits] if inits else []) + layers + ([meas] if meas else [])


def round_schedule_classes(gadgets, dur):
    classes = commuting_classes(gadgets)
    depth = 0
    time = 0
    for c in classes:
        ls = class_layers(c)
        depth += len(ls)
        time += circuit_time(ls, dur)
    return {'classes': len(classes), 'depth_ub': depth, 'time_ub': time}


import math
from collections import Counter

SOURCE_LIT = 'literature'
SOURCE_COUNT = 'counted (explicit circuit)'
SOURCE_ASSUME = 'assumption (parameter)'
SOURCE_UNRES = 'unresolved (excluded / illustrative)'

DEFAULT_PARAMS = dict(
    kappa0=1.0,      # FSC |+_L> preparation rounds = ceil(kappa0*d)   [ASSUMPTION; Kobayashi: not specified]
    kappa_in=1.0,    # switch-in rounds            = ceil(kappa_in*d)  [ASSUMPTION; Kobayashi: O(d)]
    kappa_out=1.0,   # FSC EC rounds after return  = ceil(kappa_out*d) [ASSUMPTION; Kobayashi: O(d)]
    tau_rounds=1.0,  # Litinski: 1 time step = tau_rounds*d code cycles [LITERATURE: d cycles]
    t_inj_FT=6,      # Litinski App. A (Li's injection): 3 + 3 code cycles, d-independent [LITERATURE]
    t_inj_NL=1,      # noiseless-minimal injection: 1 round [ASSUMPTION]
)


def rounds_for(model, kappa, d):
    if model == 'NL_min':
        return 1
    return int(math.ceil(kappa * d))

In [10]:
# ----------------------------------------------------------------- summaries
def summarize(rows):
    tot = Counter()
    for r in rows:
        for k, v in r['gates'].items():
            tot[k] += v
    peak = max(r['qubits'] for r in rows)
    rounds = sum(r['rounds'] for r in rows)
    tl = sum(r['time_lb'] for r in rows if r['time_lb'] is not None)
    tu = sum(r['time_ub'] for r in rows if r['time_ub'] is not None)
    v_round = sum(r['qubits'] * r['rounds'] for r in rows)
    v_tl = sum(r['qubits'] * r['time_lb'] for r in rows if r['time_lb'] is not None)
    v_tu = sum(r['qubits'] * r['time_ub'] for r in rows if r['time_ub'] is not None)
    nonclif = {k: v for k, v in tot.items() if k in NONCLIFFORD}
    tcount = 7 * tot.get('CCZ', 0) + 3 * (tot.get('CS', 0) + tot.get('CSDG', 0)) + tot.get('T', 0) + tot.get('TDG', 0)
    return dict(peak_qubits=peak, rounds=rounds, time_lb=tl, time_ub=tu, V_qubit_round=v_round,
                V_qubit_time_lb=v_tl, V_qubit_time_ub=v_tu, V_peak_x_rounds_UB=peak * rounds,
                gates=dict(tot), nonclifford=nonclif, T_count_if_decomposed=tcount)


def uncounted_costs(protocol, comparison):
    """Costs that are known to exist but are not included in the totals (listed, never summed as 0)."""
    out = [('classical decoding / control latency', 'both comparisons, both protocols')]
    if protocol == 'Kobayashi_D4':
        out.append(('JIT-decoder classical processing', 'not modelled'))
        if comparison == 'B':
            out.append(('FSC -> standard rotated surface-code patch conversion', 'UNRESOLVED: no protocol constructed'))
    else:
        out.append(('twist-based Z_m (x) Y_0 seam checks between |m> and |0>', 'O(d) measurement qubits, not constructed'))
    return out

In [11]:
# ------------------------------------------------------------------ table helpers
def jsonify_columns(df, cols=('gates', 'nonclifford')):
    """Store dict-valued columns as JSON strings (same format as the saved CSV files)."""
    import json
    for col in cols:
        if col in df.columns:
            df[col] = df[col].apply(lambda g: g if isinstance(g, str) else json.dumps({k: float(v) for k, v in g.items()}))
    return df


def identify_polynomial(values, npts=4):
    """Interpolate a polynomial in d through the first npts points (degree <= npts-1) and report
    whether it reproduces ALL given points exactly.  values: dict d -> integer.  Returns (expr, ok)."""
    import sympy
    x = sympy.Symbol('d')
    ds = sorted(values)
    if len(ds) <= npts:
        return None, False
    p = sympy.interpolate([(dd, int(round(values[dd]))) for dd in ds[:npts]], x)
    ok = all(sympy.nsimplify(p.subs(x, dd) - int(round(values[dd]))) == 0 for dd in ds)
    return sympy.expand(p), ok

### D4 ライブラリ（境界付き D4 符号・transversal $U$・code switching・疎状態シミュレータ・資源モデル）

In [12]:
"""d4_lib: D4 Clifford stabilizer code (Kobayashi-Zhu-Hsin) on the bounded square lattice, transversal U, code switching, resource model."""
import itertools, cmath
import numpy as np
from collections import Counter

R, G, B = 0, 1, 2
CNAME = "rgb"

In [13]:
# ------------------------------------------------- phase-shift operators
class PSOp:
    """Operator  O |x> = (-1)^{f(x)} |x + s>  on n qubits (computational basis x in F2^n).

    f(x) = const + sum_{i in lin} x_i + sum_{(i,j) in quad} x_i x_j   (mod 2), i<j.
    X-type Clifford stabilizers of the D4 code, Pauli X/Z strings and products
    thereof all have this form.  Composition rule (A o B)|x> = A B |x>:
        s = s_A + s_B,   f(x) = f_B(x) + f_A(x + s_B).
    """
    __slots__ = ("s", "lin", "quad", "const")

    def __init__(self, s=frozenset(), lin=frozenset(), quad=frozenset(), const=0):
        self.s = frozenset(s)
        self.lin = frozenset(lin)
        self.quad = frozenset(tuple(sorted(p)) for p in quad)
        self.const = const & 1

    def shifted_phase(self, t):
        """Return (lin, quad, const) of x -> f(x + t)."""
        lin = set(self.lin)
        const = self.const
        for i in self.lin:
            if i in t:
                const ^= 1
        for (i, j) in self.quad:
            ti, tj = i in t, j in t
            if ti and tj:
                const ^= 1
            if tj:
                lin ^= {i}
            if ti:
                lin ^= {j}
        return frozenset(lin), self.quad, const

    def __matmul__(self, other):
        lA, qA, cA = self.shifted_phase(other.s)
        return PSOp(self.s ^ other.s, lA ^ other.lin, qA ^ other.quad, cA ^ other.const)

    def phase_at(self, x):
        """f(x) for x given as a set/frozenset of 1-positions."""
        v = self.const
        for i in self.lin:
            if i in x:
                v ^= 1
        for (i, j) in self.quad:
            if i in x and j in x:
                v ^= 1
        return v

    def apply(self, x):
        """O|x> = sign |x'> ; returns (x', sign_bit)."""
        return frozenset(x) ^ self.s, self.phase_at(x)

    def support(self):
        sup = set(self.s) | set(self.lin)
        for p in self.quad:
            sup |= set(p)
        return sup


def commutator_affine(A, B):
    """A B = (-1)^{g(x)} B A on |x>.  Returns g as (lin, const) -- quadratic terms cancel."""
    AB = A @ B
    BA = B @ A
    assert AB.s == BA.s
    q = AB.quad ^ BA.quad
    assert not q, "commutator has quadratic part (should not happen)"
    return AB.lin ^ BA.lin, AB.const ^ BA.const

In [14]:
# ------------------------------------------------------------- the lattice
class D4Lattice:
    def __init__(self, L):
        assert L >= 1
        self.L = L
        self.edges = []          # (kind, x, y), kind 'h' or 'v'
        self.eidx = {}
        for y in range(L + 1):
            for x in range(L):
                self._add(('h', x, y))
        for x in range(L + 1):
            for y in range(L):
                self._add(('v', x, y))
        self.E = len(self.edges)
        self.n = 3 * self.E
        self.vertices = [(x, y) for y in range(L + 1) for x in range(L + 1)]
        self.plaquettes = [(x, y) for y in range(L) for x in range(L)]

    def _add(self, e):
        self.eidx[e] = len(self.edges)
        self.edges.append(e)

    def q(self, e, c):
        if not isinstance(e, int):
            e = self.eidx[e]
        return 3 * e + c

    # --- geometry --------------------------------------------------------
    def endpoints(self, e):
        k, x, y = self.edges[e] if isinstance(e, int) else e
        return ((x, y), (x + 1, y)) if k == 'h' else ((x, y), (x, y + 1))

    def incident(self, v):
        x, y = v
        out = []
        for e in [('h', x, y), ('h', x - 1, y), ('v', x, y), ('v', x, y - 1)]:
            if e in self.eidx:
                out.append(self.eidx[e])
        return out

    def plaq_edges(self, p):
        """(e01, e13, e02, e23) of plaquette p=(x,y)."""
        x, y = p
        ei = self.eidx
        return ei[('h', x, y)], ei[('v', x + 1, y)], ei[('v', x, y)], ei[('h', x, y + 1)]

    def plaq_boundary(self, p):
        return list(self.plaq_edges(p))

    def has_plaq(self, x, y):
        return 0 <= x < self.L and 0 <= y < self.L

    def boundary_of_edge(self, e):
        k, x, y = self.edges[e]
        L = self.L
        if k == 'v' and x == 0:
            return 'Lr'
        if k == 'h' and y == L:
            return 'Lb'
        if k == 'h' and y == 0:
            return 'Lrb'
        if k == 'v' and x == L:
            return 'Lrb'
        return None

    def boundary_edges(self, name):
        return [e for e in range(self.E) if self.boundary_of_edge(e) == name]

    # --- Z-type checks (rows of H_Z) --------------------------------------
    def z_checks(self, colours=(R, G, B)):
        """List of (label, set_of_qubits) Z-type stabilizer generators.
        Plaquettes for every colour + single-edge boundary conditions (Fig. 4 / eq. (7))."""
        rows = []
        for p in self.plaquettes:
            for c in colours:
                rows.append((('plaq', p, CNAME[c]), {self.q(e, c) for e in self.plaq_boundary(p)}))
        for e in range(self.E):
            bd = self.boundary_of_edge(e)
            if bd == 'Lr' and R in colours:
                rows.append((('bdZ', e, 'r'), {self.q(e, R)}))
            elif bd == 'Lb' and B in colours:
                rows.append((('bdZ', e, 'b'), {self.q(e, B)}))
            elif bd == 'Lrb':
                if R in colours and B in colours:
                    rows.append((('bdZ', e, 'rb'), {self.q(e, R), self.q(e, B)}))
                if G in colours:
                    rows.append((('bdZ', e, 'g'), {self.q(e, G)}))
        return rows

    # --- X-type Clifford stabilizers ----------------------------------------
    def cz_pairs(self, v, c):
        """CZ pairs (qubit, qubit) of the dressing of S^c_{X;v} (SM eq. for S_X, cup products
        evaluated with the hypercube formula; plaquettes outside the lattice are dropped)."""
        x, y = v
        pairs = []
        if c == R:      # int v u e' u e'' : v = vertex 0 of p(x,y);   CZ^{g,b}
            if self.has_plaq(x, y):
                e01, e13, e02, e23 = self.plaq_edges((x, y))
                pairs += [(self.q(e01, G), self.q(e13, B)), (self.q(e02, G), self.q(e23, B))]
        elif c == G:    # int e'' u v u e' : v = vertex 1 of p(x-1,y), vertex 2 of p(x,y-1); CZ^{r,b}
            if self.has_plaq(x - 1, y):
                e01, e13, e02, e23 = self.plaq_edges((x - 1, y))
                pairs.append((self.q(e01, R), self.q(e13, B)))
            if self.has_plaq(x, y - 1):
                e01, e13, e02, e23 = self.plaq_edges((x, y - 1))
                pairs.append((self.q(e02, R), self.q(e23, B)))
        else:           # int e' u e'' u v : v = vertex 3 of p(x-1,y-1); CZ^{r,g}
            if self.has_plaq(x - 1, y - 1):
                e01, e13, e02, e23 = self.plaq_edges((x - 1, y - 1))
                pairs += [(self.q(e01, R), self.q(e13, G)), (self.q(e02, R), self.q(e23, G))]
        return pairs

    def SX(self, v, c, dressed=True):
        s = {self.q(e, c) for e in self.incident(v)}
        quad = self.cz_pairs(v, c) if dressed else []
        # CZ applied first, then X (S = (prod X)(prod CZ)); X and CZ act on disjoint colours.
        return PSOp(s=s, quad=quad)

    def vertex_x_types(self, v, colours=(R, G, B)):
        """Allowed colour combinations at vertex v: the F2 span of colour sets whose
        X-part commutes with the boundary Z conditions.  Returns a list of colour tuples
        (a basis), derived automatically from the boundary Z checks."""
        bdrows = [rows for (lab, rows) in self.z_checks(colours) if lab[0] == 'bdZ']
        inc = self.incident(v)
        cand = []
        for k in range(1, len(colours) + 1):
            for combo in itertools.combinations(colours, k):
                s = {self.q(e, c) for e in inc for c in combo}
                if all(len(s & rw) % 2 == 0 for rw in bdrows):
                    cand.append(combo)
        # reduce to an F2 basis over colour space, preferring single colours
        basis, vecs = [], []
        for combo in sorted(cand, key=len):
            vec = np.zeros(3, dtype=np.uint8)
            vec[list(combo)] = 1
            if f2_rank(np.array(vecs + [vec])) > len(vecs):
                vecs.append(vec)
                basis.append(combo)
        return basis

    def x_generators(self, colours=(R, G, B), dressed=True):
        """List of (label, PSOp).  At a vertex where only a product of colours is allowed
        (L_rb), the generator is the ordered product S^r S^b (Fig. 4 shows S^r S^b there)."""
        gens = []
        for v in self.vertices:
            for combo in self.vertex_x_types(v, colours):
                op = PSOp()
                for c in combo:
                    op = op @ self.SX(v, c, dressed)
                gens.append((('X', v, ''.join(CNAME[c] for c in combo)), op))
        return gens

    def vec(self, qubits):
        v = np.zeros(self.n, dtype=np.uint8)
        v[list(qubits)] = 1
        return v

In [15]:
# ---------------------------------------------------------- code data
class CodeData:
    """Linear-algebra data of a (D4 or folded-surface) code on a lattice."""

    def __init__(self, lat, colours=(R, G, B), dressed=True):
        self.lat = lat
        self.colours = tuple(colours)
        self.zlabels, zr = zip(*lat.z_checks(colours))
        self.HZ = np.array([lat.vec(r) for r in zr], dtype=np.uint8)
        self.xgens = lat.x_generators(colours, dressed)
        self.S = np.array([lat.vec(op.s) for _, op in self.xgens], dtype=np.uint8)
        self.cols = sorted({lat.q(e, c) for e in range(lat.E) for c in colours})
        self.HZr, self.HZpiv = f2_rref(self.HZ)
        self.Sr, self.Spiv = f2_rref(self.S)
        # logical representatives
        CZ = f2_nullspace(self.HZ[:, self.cols])
        self.t_log = None
        for row in CZ:
            t = np.zeros(lat.n, dtype=np.uint8)
            t[self.cols] = row
            if not f2_in_rowspace(self.Sr, self.Spiv, t):
                self.t_log = t
                break
        KS = f2_nullspace(self.S[:, self.cols])
        self.z_log = None
        for row in KS:
            u = np.zeros(lat.n, dtype=np.uint8)
            u[self.cols] = row
            if not f2_in_rowspace(self.HZr, self.HZpiv, u):
                self.z_log = u
                break
        self.k = len(self.cols) - f2_rank(self.HZ[:, self.cols]) - f2_rank(self.S[:, self.cols])


def distances(cd):
    """(d_Z, d_X_shift) : min weight of a logical Z-type Pauli, and min weight of an
    X-shift lying in C_Z but outside the stabilizer shift span."""
    dz, uz = min_weight_odd(cd.S, cd.t_log, cd.cols)
    dx, tx = min_weight_odd(cd.HZ, cd.z_log, cd.cols)
    return dz, dx, uz, tx

In [16]:
# ------------------------------------------- transversal automorphism U = W V
class DiagPhase:
    """Diagonal unitary |x> -> omega^{w(x)} |x>, omega = exp(i pi/4), w integer mod 8.
    w(x) = sum_k c_k * prod_{i in mono_k} x_i  (monomials of degree <= 2)."""

    def __init__(self, terms=()):
        self.terms = list(terms)   # (coef mod 8, tuple_of_qubits)

    def value(self, x):
        v = 0
        for c, mono in self.terms:
            if all(i in x for i in mono):
                v += c
        return v % 8

    def support(self):
        s = set()
        for _, m in self.terms:
            s |= set(m)
        return s

    def terms_touching(self, qs):
        return DiagPhase([(c, m) for c, m in self.terms if set(m) & qs])


def W_operator(lat, cs_signs=(+1, -1), boundary_T=-1):
    """W of eq. (8):  prod_p CS^{r,b}_{e01,e13} (CS^{r,b}_{e02,e23})^dag  x  prod_{e in bdry_rb} (T^r_e)^dag.
    CS = diag(1,1,1,i) -> omega^{2 x y};  T^dag -> omega^{-x}."""
    terms = []
    for p in lat.plaquettes:
        e01, e13, e02, e23 = lat.plaq_edges(p)
        terms.append(((2 * cs_signs[0]) % 8, (lat.q(e01, R), lat.q(e13, B))))
        terms.append(((2 * cs_signs[1]) % 8, (lat.q(e02, R), lat.q(e23, B))))
    if boundary_T:
        for e in lat.boundary_edges('Lrb'):
            terms.append((boundary_T % 8, (lat.q(e, R),)))
    return DiagPhase(terms)


def V_map(lat, x):
    """V = prod_e CNOT^{(r,g)}_e CNOT^{(b,g)}_e :  x_g <- x_g + x_r + x_b  (set representation)."""
    x = set(x)
    out = set(x)
    for e in range(lat.E):
        qr, qg, qb = lat.q(e, R), lat.q(e, G), lat.q(e, B)
        if (qr in x) ^ (qb in x):
            out ^= {qg}
    return frozenset(out)


def V_support_map(lat, qs):
    """Qubits whose values determine (V x) restricted to qs."""
    out = set()
    for q in qs:
        e, c = divmod(q, 3)
        out.add(q)
        if c == G:
            out |= {3 * e + R, 3 * e + B}
    return out


def CZ_basis(cd):
    if not hasattr(cd, '_CZbasis'):
        nb = f2_nullspace(cd.HZ[:, cd.cols])
        full = np.zeros((nb.shape[0], cd.lat.n), dtype=np.uint8)
        full[:, cd.cols] = nb
        cd._CZbasis = full
    return cd._CZbasis


def project_CZ(cd, Q, functionals=()):
    """Exact image of C_Z under x -> (x|_Q, phi_1(x), ..., phi_k(x)) for F2-linear functionals
    phi_j (callables on a set of qubits, evaluated on basis vectors).  Returns a list of
    configurations (frozenset of Q-qubits) or, if functionals are given, (config, bits)."""
    Q = sorted(Q)
    Bs = CZ_basis(cd)
    P = Bs[:, Q]
    if functionals:
        extra = np.array([[phi(frozenset(np.nonzero(b)[0])) for phi in functionals] for b in Bs],
                         dtype=np.uint8).reshape(len(Bs), len(functionals))
        P = np.concatenate([P, extra], axis=1)
    Pr, piv = f2_rref(P)
    k = Pr.shape[0]
    assert k <= 24, f"local projection too large ({k})"
    out = []
    nq = len(Q)
    for bits in itertools.product((0, 1), repeat=k):
        v = np.zeros(P.shape[1], dtype=np.uint8)
        for b, row in zip(bits, Pr):
            if b:
                v ^= row
        cfg = frozenset(Q[i] for i in np.nonzero(v[:nq])[0])
        out.append((cfg, tuple(int(z) for z in v[nq:])) if functionals else cfg)
    return out


def check_conjugation(cd, Wop, gen_op, target_op):
    """Exhaustively check  U S U^dag == S'  on C_Z, where U = W V.
    Returns (ok, n_local_configs)."""
    lat = cd.lat
    As = V_map(lat, gen_op.s)
    if As != target_op.s:
        return False, 0
    # qubits on which the phase difference can depend
    Wt = Wop.terms_touching(set(As))
    Q = set(Wt.support()) | V_support_map(lat, gen_op.support()) | target_op.support() | set(As)
    configs = project_CZ(cd, Q)
    for x in configs:
        # omega^{ w(x+As) - w(x) + 4 f(A x) - 4 f'(x) }   (only W-terms touching As contribute)
        d = Wt.value(x ^ As) - Wt.value(x) + 4 * gen_op.phase_at(V_map(lat, x)) - 4 * target_op.phase_at(x)
        if d % 8:
            return False, len(configs)
    return True, len(configs)


def stabilizer_path_phase(cd, x, target):
    """Given x and target = x + (sum of generator shifts), apply generators to |x> and
    return the accumulated sign bit, i.e. psi(target) = (-1)^bit psi(x) for code states."""
    shifts = cd.S
    delta = cd.lat.vec(set(x) ^ set(target))
    coeff = f2_solve(shifts, delta)
    if coeff is None:
        return None
    bit = 0
    cur = frozenset(x)
    for i in np.nonzero(coeff)[0]:
        op = cd.xgens[i][1]
        cur, b = op.apply(cur)
        bit ^= b
    assert cur == frozenset(target)
    return bit


def logical_phases(cd, Wop):
    """Eigenphases (in units of pi/4) of U on |0_L>, |1_L>, assuming U preserves the code space
    (checked separately).  Uses x0 = 0 and x1 = t_log."""
    lat = cd.lat
    res = []
    for x in [frozenset(), frozenset(np.nonzero(cd.t_log)[0])]:
        Ax = V_map(lat, x)
        bit = stabilizer_path_phase(cd, x, Ax)
        assert bit is not None, "V does not map the coset to itself"
        res.append((Wop.value(Ax) + 4 * bit) % 8)
    return res

In [17]:
# ------------------------------------------- per-configuration protocol phase
def lrb_vertices(lat):
    vs = set()
    for e in lat.boundary_edges('Lrb'):
        vs |= set(lat.endpoints(e))
    return vs


def cprime(lat, a):
    """0-cochain c' with  delta c' = a_r + a_b  and c' = 0 on L_rb vertices (BFS).
    a: set of qubits (configuration of r,b qubits).  Returns dict vertex -> bit.
    Raises if a_r + a_b is not exact relative to L_rb."""
    from collections import deque
    val = {v: 0 for v in lrb_vertices(lat)}
    dq = deque(val)
    while dq:
        u = dq.popleft()
        for e in lat.incident(u):
            p, q = lat.endpoints(e)
            w = q if p == u else p
            bit = val[u] ^ (lat.q(e, R) in a) ^ (lat.q(e, B) in a)
            if w in val:
                if val[w] != bit:
                    raise ValueError("a_r + a_b not exact")
            else:
                val[w] = bit
                dq.append(w)
    return val


def g_phase_poly(lat, v):
    """CZ^{r,b} pairs of S^g_{X;v}  (f^g_v)."""
    return lat.cz_pairs(v, G)


def protocol_phase(lat, Wop, a, gverts):
    """Phase exponent (mod 8, units of pi/4) acquired by FSC configuration a through
    switch-in (S^g measurement) -> U -> Z^g measurement -> S^g correction -> Pauli frame,
    i.e.  F(a) = w(a) + 4 * sum_v c'_v(a) f^g_v(a)."""
    cp = cprime(lat, a)
    phi = 0
    for v in gverts:
        if cp[v]:
            for (i, j) in g_phase_poly(lat, v):
                if i in a and j in a:
                    phi ^= 1
    return (Wop.value(a) + 4 * phi) % 8


def check_protocol_invariance(lat, fsc, Wop, gverts, gen_op):
    """Exhaustive local proof that F(a + t) = F(a) (mod 8) for all a in C_FSC, t = shift of gen_op."""
    t = frozenset(gen_op.s)
    # c'(t) (linear in a)
    ct = cprime(lat, t)
    N2 = {v for v in gverts if ct[v]}
    N1 = {v for v in gverts if set(q for pr in g_phase_poly(lat, v) for q in pr) & t}
    Wt = Wop.terms_touching(set(t))
    Q = set(Wt.support()) | set(t)
    for v in N1 | N2:
        Q |= {q for pr in g_phase_poly(lat, v) for q in pr}
    # paths from a reference vertex w0 to each w in N1 (BFS on the lattice graph)
    w0 = min(N1 | N2 | {lat.endpoints(next(iter(t)) // 3)[0]})
    paths = {}
    from collections import deque
    prev = {w0: None}
    dq = deque([w0])
    while dq:
        u = dq.popleft()
        for e in lat.incident(u):
            p, q = lat.endpoints(e)
            w = q if p == u else p
            if w not in prev:
                prev[w] = (u, e)
                dq.append(w)
    for w in N1:
        es = []
        cur = w
        while prev[cur] is not None:
            u, e = prev[cur]
            es.append(e)
            cur = u
        paths[w] = es
        Q |= {lat.q(e, c) for e in es for c in (R, B)}
    beta_fn = lambda b: cprime(lat, b)[w0]          # c'_{w0}(a) is F2-linear in a
    configs = project_CZ(fsc, Q, functionals=[beta_fn])
    for a, (beta,) in configs:
        if True:
            cpa = {}
            for w in N1:
                bit = beta
                for e in paths[w]:
                    bit ^= (lat.q(e, R) in a) ^ (lat.q(e, B) in a)
                cpa[w] = bit
            at = a ^ t
            d = Wt.value(at) - Wt.value(a)
            phi = 0
            for w in N1:
                if cpa[w]:
                    for (i, j) in g_phase_poly(lat, w):
                        phi ^= ((i in at) and (j in at)) ^ ((i in a) and (j in a))
            for w in N2:
                for (i, j) in g_phase_poly(lat, w):
                    phi ^= ((i in at) and (j in at))
            if (d + 4 * phi) % 8:
                return False, len(configs)
    return True, len(configs)


OMEGA = cmath.exp(1j * cmath.pi / 4)


class SparseState:
    def __init__(self, amps):
        self.a = {k: v for k, v in amps.items() if abs(v) > 1e-14}

    def norm2(self):
        return sum(abs(v) ** 2 for v in self.a.values())

    def normalize(self):
        n = self.norm2() ** 0.5
        self.a = {k: v / n for k, v in self.a.items()}
        return self

    def apply_psop(self, op):
        out = {}
        for x, v in self.a.items():
            y, b = op.apply(x)
            out[y] = out.get(y, 0) + (-v if b else v)
        return SparseState(out)

    def apply_diag(self, D):
        return SparseState({x: v * OMEGA ** D.value(x) for x, v in self.a.items()})

    def apply_perm(self, f):
        out = {}
        for x, v in self.a.items():
            y = f(x)
            out[y] = out.get(y, 0) + v
        return SparseState(out)

    def project(self, op, sign):
        """(1 + sign*O)/2 |psi>  (unnormalised)."""
        Opsi = self.apply_psop(op).a
        out = {}
        for k in set(self.a) | set(Opsi):
            out[k] = 0.5 * (self.a.get(k, 0) + sign * Opsi.get(k, 0))
        return SparseState(out)

    def expect(self, op):
        Opsi = self.apply_psop(op).a
        return sum(np.conj(self.a.get(k, 0)) * v for k, v in Opsi.items())

    def measure_op(self, op, rng, forced=None):
        """Projective measurement of a Hermitian involution O; returns (outcome bit, state)."""
        p0 = self.project(op, +1)
        n0 = p0.norm2()
        if forced is None:
            bit = 0 if rng.random() < n0 else 1
        else:
            bit = forced
        st = p0 if bit == 0 else self.project(op, -1)
        prob = n0 if bit == 0 else 1 - n0
        return bit, st.normalize() if prob > 1e-12 else st, prob

    def measure_z(self, q, rng, forced=None):
        p1 = sum(abs(v) ** 2 for x, v in self.a.items() if q in x)
        bit = (1 if rng.random() < p1 else 0) if forced is None else forced
        st = SparseState({x: v for x, v in self.a.items() if (q in x) == bool(bit)})
        prob = p1 if bit else 1 - p1
        return bit, st.normalize() if prob > 1e-12 else st, prob

    def overlap(self, other):
        return sum(np.conj(v) * other.a.get(k, 0) for k, v in self.a.items())


def coset_state(cd, rep, phases=True):
    """Code state supported on rep + span(shifts):  psi(rep)=1, psi(x+s_v)=(-1)^{f_v(x)} psi(x)."""
    amps = {frozenset(rep): 1.0}
    # breadth-first application of generators
    frontier = [frozenset(rep)]
    while frontier:
        new = []
        for x in frontier:
            for _, op in cd.xgens:
                y, b = op.apply(x)
                if y not in amps:
                    amps[y] = amps[x] * (-1 if b else 1)
                    new.append(y)
                else:
                    if phases:
                        assert abs(amps[y] - amps[x] * (-1 if b else 1)) < 1e-9, "inconsistent code state"
        frontier = new
    return SparseState(amps).normalize()




def frozen_qubits(lat):
    """Qubits fixed to |0> by single-qubit boundary Z stabilizers (L_r: r, L_b: b, L_rb: g)."""
    fz = set()
    for lab, rows in lat.z_checks():
        if lab[0] == 'bdZ' and len(rows) == 1:
            fz |= rows
    return fz


class D4Protocol:
    def __init__(self, L, prune_frozen=True):
        self.L = L
        self.lat = lat = D4Lattice(L)
        self.frozen = frozen_qubits(lat) if prune_frozen else set()
        self.prune = prune_frozen
        self.d4 = CodeData(lat)
        self.fsc = CodeData(lat, (R, B), dressed=False)
        # ancilla allocation: one ancilla per stabilizer generator label (D4 superset)
        self.anc = {}
        nxt = lat.n
        labels = [lab for lab, _ in self.d4.xgens] + list(self.d4.zlabels)
        labels += [lab for lab, _ in self.fsc.xgens] + list(self.fsc.zlabels)
        for lab in labels:
            key = self._anc_key(lab)
            if key not in self.anc and not self._is_frozen_check(lab):
                self.anc[key] = nxt
                nxt += 1
        self.gverts = [lab[1] for lab, op in self.d4.xgens if lab[2] == 'g']

    @staticmethod
    def _anc_key(lab):
        return lab

    def _is_frozen_check(self, lab):
        return lab[0] == 'bdZ' and lab[2] in ('r', 'b', 'g') and self.prune

    def data_qubits(self, colours):
        return {self.lat.q(e, c) for e in range(self.lat.E) for c in colours} - self.frozen

    # ---------------------------------------------------------- gadgets
    def _keep_pair(self, i, j):
        return i not in self.frozen and j not in self.frozen

    def x_gadget(self, lab, dressed):
        _, v, cols = lab
        parts = []
        for ch in cols:
            c = CNAME.index(ch)
            xq = {self.lat.q(e, c) for e in self.lat.incident(v)} - self.frozen
            pairs = [p for p in self.lat.cz_pairs(v, c) if self._keep_pair(*p)] if dressed else []
            parts.append((xq, pairs))
        return clifford_x_gadget(self.anc[lab], parts)

    def z_gadget(self, lab, rows):
        return z_check_gadget(self.anc[lab], set(rows) - self.frozen)

    def round_gadgets(self, code, which='all'):
        """Gadgets of one stabilizer round.  code: 'D4' or 'FSC'.
        which: 'all' | 'X' | 'Z' | 'g' (D4 S^g only)."""
        cd = self.d4 if code == 'D4' else self.fsc
        dressed = code == 'D4'
        gads = []
        if which in ('all', 'X', 'g'):
            for lab, op in cd.xgens:
                if which == 'g' and lab[2] != 'g':
                    continue
                gads.append(self.x_gadget(lab, dressed))
        if which in ('all', 'Z'):
            for lab, (l2, rows) in zip(cd.zlabels, self.lat.z_checks(cd.colours)):
                if self._is_frozen_check(lab):
                    continue
                gads.append(self.z_gadget(lab, rows))
        return gads

    # ---------------------------------------------------------- transversal U
    def U_gates(self):
        lat, fz = self.lat, self.frozen
        gates = []
        for e in range(lat.E):
            qr, qg, qb = lat.q(e, R), lat.q(e, G), lat.q(e, B)
            for ctrl in (qr, qb):
                if ctrl in fz:
                    continue
                if qg in fz:
                    continue   # L_rb: CNOT(r,g) CNOT(b,g) = identity on Z^r Z^b = +1 (g removed)
                gates.append(('CNOT', ctrl, qg))
        for p in lat.plaquettes:
            e01, e13, e02, e23 = lat.plaq_edges(p)
            for (a, b, name) in [(lat.q(e01, R), lat.q(e13, B), 'CS'), (lat.q(e02, R), lat.q(e23, B), 'CSDG')]:
                if a in fz or b in fz:
                    continue
                gates.append((name, a, b))
        for e in lat.boundary_edges('Lrb'):
            gates.append(('TDG', lat.q(e, R)))
        return gates

    # ---------------------------------------------------------- switch-out
    def switch_out_gates(self, worst_case=True):
        lat = self.lat
        gq = sorted(self.data_qubits((G,)))
        gates = [('MEAS_Z', q) for q in gq]
        cz = []
        for v in self.gverts:
            for p in lat.cz_pairs(v, G):
                if self._keep_pair(*p):
                    cz.append(('CZ',) + tuple(p))
        return gates, cz      # cz: worst-case correction (all vertices in kappa)

    def qubit_sets(self):
        fsc_data = self.data_qubits((R, B))
        d4_data = self.data_qubits((R, G, B))
        fsc_anc = {self.anc[l] for l, _ in self.fsc.xgens} | {self.anc[l] for l in self.fsc.zlabels
                                                              if not self._is_frozen_check(l)}
        d4_anc = {self.anc[l] for l, _ in self.d4.xgens} | {self.anc[l] for l in self.d4.zlabels
                                                            if not self._is_frozen_check(l)}
        return dict(fsc_data=fsc_data, d4_data=d4_data, fsc_anc=fsc_anc, d4_anc=d4_anc)

In [18]:
# ----------------------------------------------------------------- Kobayashi (D4)
class D4CostData:
    """All d-dependent counts of the D4 protocol, computed once per L from explicit circuits."""

    def __init__(self, L, dur=None, decomposed=False):
        dur = dur or DUR_UNIT
        P = D4Protocol(L)
        self.L = L
        qs = P.qubit_sets()
        self.n = {k: len(v) for k, v in qs.items()}
        self.n_total = len(qs['d4_data'] | qs['d4_anc'])
        dec = decompose if decomposed else (lambda g: g)
        gad = {
            'D4': [dec(g) for g in P.round_gadgets('D4')],
            'D4g': [dec(g) for g in P.round_gadgets('D4', 'g')],
            'FSC': P.round_gadgets('FSC'),
        }
        self.round = {}
        for k, g in gad.items():
            ub = round_schedule_classes(g, dur)
            lb = round_lower_bound(g, dur)
            self.round[k] = dict(time_lb=lb['time_lb'], time_ub=ub['time_ub'], depth_lb=lb['depth_lb'],
                                 depth_ub=ub['depth_ub'], classes=ub['classes'],
                                 gates=Counter(gate_counts([x for gg in g for x in gg])),
                                 n_anc=len(g))
        self.n_anc_g = len(gad['D4g'])
        U = dec(P.U_gates())
        Ul = class_layers([U])              # all U gates mutually commute (validated)
        self.U = dict(gates=Counter(gate_counts(U)), time=circuit_time(Ul, dur), depth=len(Ul))
        meas, cz = P.switch_out_gates()
        czl = class_layers([cz]) if cz else []
        ml = [meas]
        self.out = dict(gates_worst=Counter(gate_counts(meas + cz)),
                        gates_expected=Counter({**gate_counts(meas), 'CZ': len(cz) / 2}),
                        time=circuit_time(ml + czl, dur), depth=len(ml) + len(czl))
        self.x_corr = dict(time=dur['X'], gates_worst=Counter({'X': self.n['fsc_data']}))
        self.init = Counter({'INIT_X': self.n['fsc_data'], 'INIT_Z': len(qs['d4_data']) - self.n['fsc_data']})


def d4_stages(cd, model, params=DEFAULT_PARAMS, comparison='A', data_only=False):
    d = cd.L
    n = cd.n
    fsc_q = n['fsc_data'] + (0 if data_only else n['fsc_anc'])
    d4_q = n['d4_data'] + (0 if data_only else n['d4_anc'])
    rows = []

    def add(stage, qubits, rounds, rkey, extra_time, gates, source, note=''):
        rt = cd.round[rkey] if rkey else dict(time_lb=0, time_ub=0)
        rows.append(dict(protocol='Kobayashi_D4', comparison=comparison, model=model, d=d, stage=stage,
                         qubits=qubits, rounds=rounds,
                         time_lb=rounds * rt['time_lb'] + extra_time,
                         time_ub=rounds * rt['time_ub'] + extra_time,
                         gates=dict(gates), source=source, note=note))

    r0 = rounds_for(model, params['kappa0'], d)
    g0 = Counter(cd.init)
    for k, v in cd.round['FSC']['gates'].items():
        g0[k] += v * r0
    g0['X'] += cd.x_corr['gates_worst']['X']
    add('S0 FSC |+_L> preparation', fsc_q, r0, 'FSC', cd.x_corr['time'], g0,
        SOURCE_ASSUME if model == 'FT_ref' else SOURCE_COUNT,
        'kappa0*d rounds (not specified in Kobayashi); X-correction worst case counted')
    if model == 'NL_min':
        r1, key = 1, 'D4g'
        q1 = n['d4_data'] + (0 if data_only else cd.n_anc_g)
        note1 = 'noiseless minimum: one round of S^g_X measurements only'
    else:
        r1, key = rounds_for(model, params['kappa_in'], d), 'D4'
        q1 = d4_q
        note1 = 'O(d) rounds of all D4 stabilizers (Kobayashi step 2); kappa_in*d assumed'
    g1 = Counter()
    for k, v in cd.round[key]['gates'].items():
        g1[k] += v * r1
    add('S1 switch-in (gauging) to D4', q1, r1, key, 0, g1,
        SOURCE_ASSUME if model == 'FT_ref' else SOURCE_COUNT, note1)
    add('S2 transversal U = W V (logical T^dag)', n['d4_data'], 0, None, cd.U['time'], cd.U['gates'],
        SOURCE_COUNT, 'constant depth; gates from eq. (8)')
    add('S3 switch-out: Z^g meas + S^g correction', n['d4_data'], 0, None, cd.out['time'],
        cd.out['gates_worst'], SOURCE_COUNT, 'CZ correction counted worst case (expected = half)')
    r4 = 0 if model == 'NL_min' else rounds_for(model, params['kappa_out'], d)
    g4 = Counter()
    for k, v in cd.round['FSC']['gates'].items():
        g4[k] += v * r4
    add('S4 FSC error-correction rounds', fsc_q, r4, 'FSC', 0, g4,
        SOURCE_ASSUME if model == 'FT_ref' else SOURCE_COUNT,
        'O(d) rounds (Kobayashi step 4); kappa_out*d assumed; 0 in the noiseless minimum')
    add('S5 logical X frame: T^dag|+> -> T|+>', 0, 0, None, 0, Counter(), SOURCE_COUNT,
        'Pauli frame update, no physical cost')
    # NOTE (comparison B): the FSC -> standard rotated-patch conversion is UNRESOLVED.  It is deliberately
    # NOT added as a zero-cost stage; it is reported separately by uncounted_costs().
    return rows



def d4_conversion_illustrative(cd, model, params=DEFAULT_PARAMS, data_only=False):
    """ILLUSTRATIVE ONLY: conversion modelled as one lattice-surgery time step (tau) during which the
    FSC plus 2 tiles (target patch + ancilla strip) are active.  Not taken from literature."""
    d = cd.L
    r = rounds_for(model, params['tau_rounds'], d)
    q = cd.n['fsc_data'] + (0 if data_only else cd.n['fsc_anc']) + 2 * (d * d if data_only else 2 * d * d)
    return dict(protocol='Kobayashi_D4', comparison='B_illustrative', model=model, d=d,
                stage='S6 conversion (ILLUSTRATIVE: 1 tau lattice surgery)', qubits=q, rounds=r,
                time_lb=None, time_ub=None, gates={}, source=SOURCE_UNRES,
                note='illustrative assumption, not a validated protocol')

In [19]:
# ---------------------------------------------------------- consistency checks of a code
def code_consistency(cd):
    """Exact F2 checks of a (possibly non-commuting) Clifford stabilizer code on its Z-subspace C_Z:
      shift_in_CZ : every X-type generator preserves C_Z (its X part commutes with all Z checks)
      noncommuting_pairs : generator pairs that do NOT commute on C_Z
      bad_squares : generators with S^2 != 1 on C_Z
      k : number of logical qubits = dim C_Z - rank(shifts)   (generators have independent shifts)"""
    lat = cd.lat
    HZ, S = cd.HZ, cd.S
    res = {}
    res['shift_in_CZ'] = not ((HZ.astype(int) @ S.T.astype(int)) % 2).any()
    supp = [op.support() for _, op in cd.xgens]
    bad = []
    for i in range(len(cd.xgens)):
        for j in range(i + 1, len(cd.xgens)):
            if not (supp[i] & supp[j]):
                continue
            lin, const = commutator_affine(cd.xgens[i][1], cd.xgens[j][1])
            if const or not f2_in_rowspace(cd.HZr, cd.HZpiv, lat.vec(lin)):
                bad.append((cd.xgens[i][0], cd.xgens[j][0]))
    res['noncommuting_pairs'] = bad
    sq = []
    for lab, op in cd.xgens:
        s2 = op @ op
        if s2.s or s2.quad or s2.const or not f2_in_rowspace(cd.HZr, cd.HZpiv, lat.vec(s2.lin)):
            sq.append(lab)
    res['bad_squares'] = sq
    res['n_qubits'] = len(cd.cols)
    res['dim_CZ'] = len(cd.cols) - f2_rank(HZ[:, cd.cols])
    res['rank_shifts'] = f2_rank(S[:, cd.cols])
    res['n_xgens'] = len(cd.xgens)
    res['k'] = res['dim_CZ'] - res['rank_shifts']
    res['shifts_independent'] = res['rank_shifts'] == res['n_xgens']
    return res


def U_target(lat, lab):
    """Image of generator `lab` under U predicted by SM eq. (permutingstabilizers):
    S^r -> S^r S^g, S^g -> S^g, S^b -> S^g S^b, and (boundary L_rb) S^r S^b -> S^r S^b."""
    _, v, cols = lab
    if cols == 'rb':
        return lat.SX(v, R) @ lat.SX(v, B)
    op = PSOp()
    for ch in cols:
        c = CNAME.index(ch)
        for cc in ([c] if c == G else [c, G]):
            op = op @ lat.SX(v, cc)
    return op

In [20]:
# ---------------------------------------------------------- Knill-Laflamme distance lower bound
def _milp_binary(c_obj, A_eq_mod2, rhs_mod2, extra_ub=None, fixed_zero=()):
    """min c.x over binary x subject to A x = rhs (mod 2) (rows of A_eq_mod2), and optional
    linear constraints extra_ub = [(row, ub), ...] (row.x <= ub).
    Returns (value, x) if an optimum was found, (None, None) only if the problem is proven
    infeasible (scipy status 2).  Any other status (time/iteration limit, unbounded, numerical
    failure) raises RuntimeError, so that a solver failure is never mistaken for "no solution"."""
    from scipy.optimize import milp, LinearConstraint, Bounds
    A = np.array(A_eq_mod2, dtype=float)
    m, n = A.shape
    nv = n + m
    c = np.concatenate([np.asarray(c_obj, float), np.zeros(m)])
    cons = [LinearConstraint(np.concatenate([A, -2 * np.eye(m)], axis=1), rhs_mod2, rhs_mod2)]
    for row, ub in (extra_ub or []):
        cons.append(LinearConstraint(np.concatenate([np.asarray(row, float), np.zeros(m)])[None, :], -np.inf, ub))
    ub_ = np.concatenate([np.ones(n), np.full(m, np.inf)])
    for i in fixed_zero:
        ub_[i] = 0
    res = milp(c, constraints=cons, integrality=np.ones(nv), bounds=Bounds(np.zeros(nv), ub_))
    if res.status == 2:
        return None, None
    if res.status != 0:
        raise RuntimeError(f'MILP solver did not finish (status {res.status}: {res.message})')
    return round(res.fun), np.round(res.x[:n]).astype(np.uint8)


def low_weight_stabilizer_shifts(cd, wmax, frozen=()):
    """Exact enumeration (MILP with no-good cuts) of all t in span(S) with 1 <= |t| <= wmax,
    where span(S) = C_Z  intersect  z_log^perp  (k = 1).  Frozen qubits are excluded
    (an X on a frozen qubit violates its single-qubit Z check, so t would leave C_Z).
    Returns (found, stop) where stop records why the enumeration ended:
    'infeasible' (no further t with 1 <= |t| <= wmax) or 'min weight > wmax'."""
    cols = [c for c in cd.cols if c not in set(frozen)]
    H = np.vstack([cd.HZ[:, cols], cd.z_log[cols][None, :]])
    H = H[np.any(H, axis=1)]
    rhs = np.zeros(len(H))
    n = len(cols)
    cuts = [(np.ones(n), wmax), (-np.ones(n), -1)]
    found = []
    while True:
        val, x = _milp_binary(np.ones(n), H, rhs, extra_ub=cuts)
        if val is None:
            stop = 'infeasible'
            break
        if val > wmax:
            stop = 'min weight > wmax'
            break
        sup = np.nonzero(x)[0]
        t = np.zeros(cd.lat.n, dtype=np.uint8)
        t[[cols[i] for i in sup]] = 1
        found.append(t)
        row = -np.ones(n)
        row[sup] = 1
        cuts.append((row, len(sup) - 1))                  # exclude exactly this vector
    return found, stop


def kl_distance_lower_bound(cd, wmax, frozen=()):
    r"""Rigorous lower-bound argument for the Knill-Laflamme distance of the (non-Pauli) code cd.

    For a Pauli E = X^t Z^u (up to phase):
      * t not in C_Z           -> E maps the code space to an orthogonal space: detectable;
      * t in the logical coset -> |t| >= X-shift distance (checked separately);
      * t in span(S)           -> X^t = S_c F_c with S_c a stabilizer and F_c = (-1)^{f_c} diagonal,
        so P E P = P F_c Z^u P.  Expanding (-1)^{f_c} in Z-type Paulis Z^v with v supported on
        Q_f = supp(f_c), a violation of KL requires some v + u to be a logical Z operator l.
        Hence |E| >= |t| + |u \ t| >= |t| + min_l |l \ (Q_f u t)|  =: LB_t.
    Returns (min over enumerated t of LB_t, ts, details, stop) where stop is the termination
    reason of the enumeration (see low_weight_stabilizer_shifts).  t = 0 is covered by d_Z."""
    lat = cd.lat
    ts, stop = low_weight_stabilizer_shifts(cd, wmax, frozen)
    cols = [c for c in cd.cols if c not in set(frozen)]
    Srows = cd.S[:, cols]
    Srows = Srows[np.any(Srows, axis=1)]
    best, details = None, []
    for t in ts:
        coef = f2_solve(cd.S, t)
        op = PSOp()
        for i in np.nonzero(coef)[0]:
            op = op @ cd.xgens[i][1]
        assert op.s == frozenset(np.nonzero(t)[0])
        Qf = {q for pr in op.quad for q in pr} | set(op.lin) | set(np.nonzero(t)[0])
        cost = np.array([0.0 if c in Qf else 1.0 for c in cols])
        A = np.vstack([Srows, cd.t_log[cols][None, :]])
        rhs = np.concatenate([np.zeros(len(Srows)), [1.0]])
        val, _ = _milp_binary(cost, A, rhs)
        assert val is not None                            # a logical Z always exists
        lb = int(t.sum()) + val
        details.append(dict(weight_t=int(t.sum()), Qf_size=len(Qf), LB=lb))
        best = lb if best is None else min(best, lb)
    return best, ts, details, stop

In [21]:
# ---------------------------------------------------------- full resource tables (used by main + comparison)
def d4_resource_tables(D_LIST, dur=None, params=None, verbose=True):
    """Per-stage and summary tables of the Kobayashi protocol for every d in D_LIST, gate set
    ('primitive' | 'clifford_t'), counting variant (data_only), time model and comparison A/B.
    Returns (stage_df, sum_df, blocks)."""
    import pandas as pd, time
    dur = dur or DUR_UNIT
    params = params or DEFAULT_PARAMS
    records, summaries, blocks = [], [], {}
    for d in D_LIST:
        t0 = time.time()
        for gs in ['primitive', 'clifford_t']:
            cd = D4CostData(d, dur, decomposed=(gs == 'clifford_t'))
            for data_only in [False, True]:
                for model in ['FT_ref', 'NL_min']:
                    for comp in ['A', 'B']:
                        st = d4_stages(cd, model, params, comp, data_only=data_only)
                        for r in st:
                            r.update(gateset=gs, data_only=data_only)
                        records += st
                        s = summarize(st)
                        s.update(protocol='Kobayashi_D4', d=d, model=model, comparison=comp, gateset=gs,
                                 data_only=data_only)
                        summaries.append(s)
            blocks[f'{d}|{gs}'] = dict(d=d, gateset=gs, n=cd.n, n_total=cd.n_total, n_anc_g=cd.n_anc_g,
                                       round={k: {kk: (dict(vv) if isinstance(vv, Counter) else vv) for kk, vv in v.items()}
                                              for k, v in cd.round.items()},
                                       U_time=cd.U['time'], U_gates=dict(cd.U['gates']), out_time=cd.out['time'],
                                       out_gates=dict(cd.out['gates_worst']), xcorr_time=cd.x_corr['time'])
        if verbose:
            print(f'Kobayashi d={d}: {time.time() - t0:.1f}s')
    return jsonify_columns(pd.DataFrame(records)), jsonify_columns(pd.DataFrame(summaries)), blocks

In [22]:
print('library loaded')

library loaded


In [23]:
import time, itertools
VAL = {}
def record(name, passed, detail=''):
    detail = str(detail).replace('\n', ' ')
    VAL[name] = dict(status='PASS' if passed else 'FAIL', detail=detail)
    print(('PASS ' if passed else 'FAIL ') + name + (f'  | {detail}' if detail else ''))
QUICK = False
L_MAX_INVARIANCE = 5 if QUICK else 8
rng = np.random.default_rng(SEED)

## V1. 安定化子の定義と [KZH] Fig. 4 の照合

Fig. 4（$L=9$）から読み取れる事項を、コードから得た構造と照合する。
* 左境界 $\mathcal L_r$ 上：緑・青の $X$ 安定化子（3辺）。赤は無し。
* 上境界 $\mathcal L_b$ 上：赤・緑。左上角：緑（2辺）。
* 下・右境界 $\mathcal L_{rb}$：赤青の積 $XX$。右下角も $XX$（2辺）。
* 境界 $Z$：$\mathcal L_r$ に $Z^r$、$\mathcal L_b$ に $Z^b$、$\mathcal L_{rb}$ に $Z^rZ^b$ と $Z^g$。
* バルクの CZ：$S^r$ は右上 plaquette に CZ$^{g,b}$ 2本、$S^g$ は左上・右下 plaquette に CZ$^{r,b}$ 各1本、$S^b$ は左下 plaquette に CZ$^{r,g}$ 2本。

In [24]:
lat = D4Lattice(9); L = 9
exp = {'bulk': [('r',), ('g',), ('b',)], 'Lr': [('g',), ('b',)], 'Lb': [('r',), ('g',)], 'TL': [('g',)],
       'Lrb': [('r', 'b')], 'BL': [], 'TR': []}
def vclass(v):
    x, y = v
    if v == (0, 0): return 'BL'
    if v == (0, L): return 'TL'
    if v == (L, L): return 'TR'
    if 0 < x < L and 0 < y < L: return 'bulk'
    if x == 0: return 'Lr'
    if y == L: return 'Lb'
    return 'Lrb'
ok = True
for v in lat.vertices:
    got = [tuple(CNAME[c] for c in combo) for combo in lat.vertex_x_types(v)]
    if sorted(got) != sorted(exp[vclass(v)]):
        ok = False; print('mismatch at', v, got)
record('V1a boundary X-generator types match Fig. 4', ok)
zl = lat.z_checks()
bd = Counter((lab[2], lat.boundary_of_edge(lab[1])) for lab, _ in zl if lab[0] == 'bdZ')
exp_bd = {('r', 'Lr'): L, ('b', 'Lb'): L, ('rb', 'Lrb'): 2 * L, ('g', 'Lrb'): 2 * L}
record('V1b boundary Z checks (Z^r on L_r, Z^b on L_b, Z^rZ^b and Z^g on L_rb)', dict(bd) == exp_bd, dict(bd))
def rel_plaq(v, c):
    """plaquette offsets (relative to v) that carry the CZ dressing of S^c_v"""
    out = []
    for (i, j) in lat.SX(v, c).quad:
        es = {i // 3, j // 3}
        for p in lat.plaquettes:
            if es <= set(lat.plaq_edges(p)):
                out.append((p[0] - v[0], p[1] - v[1]))
    return Counter(out)
v0 = (4, 4)
ok = (rel_plaq(v0, R) == Counter({(0, 0): 2}) and rel_plaq(v0, G) == Counter({(-1, 0): 1, (0, -1): 1})
      and rel_plaq(v0, B) == Counter({(-1, -1): 2}))
record('V1c bulk CZ positions (S^r: upper-right x2, S^g: upper-left + lower-right, S^b: lower-left x2)', ok,
       {c: dict(rel_plaq(v0, c)) for c in (R, G, B)})
colpairs = {c: {tuple(sorted((i % 3, j % 3))) for (i, j) in lat.SX(v0, c).quad} for c in (R, G, B)}
record('V1d CZ colour pairs (S^r: g-b, S^g: r-b, S^b: r-g)', colpairs == {R: {(G, B)}, G: {(R, B)}, B: {(R, G)}})

PASS V1a boundary X-generator types match Fig. 4
PASS V1b boundary Z checks (Z^r on L_r, Z^b on L_b, Z^rZ^b and Z^g on L_rb)  | {('rb', 'Lrb'): 18, ('g', 'Lrb'): 18, ('b', 'Lb'): 9, ('r', 'Lr'): 9}
PASS V1c bulk CZ positions (S^r: upper-right x2, S^g: upper-left + lower-right, S^b: lower-left x2)  | {0: {(0, 0): 2}, 1: {(0, -1): 1, (-1, 0): 1}, 2: {(-1, -1): 2}}
PASS V1d CZ colour pairs (S^r: g-b, S^g: r-b, S^b: r-g)


## V2. 符号空間の整合性（$\mathbb F_2$ 上の厳密判定）

$X$ 型 Clifford 安定化子 $A,B$ について $AB=(-1)^{g(x)}BA$ の $g$ はアフィン関数になる（2次項は相殺）。$C_Z$ 上で $g\equiv 0$ ⇔ 定数項0かつ線形部分が $H_Z$ の行空間に属する。
**負の対照**：FSC（$g$ を除いた2層）に CZ 修飾付きの生成元を使うと、$g$ 量子ビットの $Z$ 安定化子が無いため非可換になるはず → 検出できることを確認。

In [25]:
for L in range(1, 7):
    cd = CodeData(D4Lattice(L))
    r = code_consistency(cd)
    record(f'V2 D4 L={L}: shifts in C_Z, commuting on C_Z, S^2=1, k=1',
           r['shift_in_CZ'] and not r['noncommuting_pairs'] and not r['bad_squares'] and r['k'] == 1 and r['shifts_independent'],
           {k: (len(v) if isinstance(v, list) else v) for k, v in r.items()})
    rf = code_consistency(CodeData(D4Lattice(L), (R, B), dressed=False))
    record(f'V2 FSC L={L}: CSS code, k=1', rf['shift_in_CZ'] and not rf['noncommuting_pairs'] and rf['k'] == 1)
neg = code_consistency(CodeData(D4Lattice(3), (R, B), dressed=True))
record('V2 negative control: dressed generators without g-qubit Z checks are detected as non-commuting',
       len(neg['noncommuting_pairs']) > 0, f"{len(neg['noncommuting_pairs'])} non-commuting pairs")

PASS V2 D4 L=1: shifts in C_Z, commuting on C_Z, S^2=1, k=1  | {'shift_in_CZ': True, 'noncommuting_pairs': 0, 'bad_squares': 0, 'n_qubits': 12, 'dim_CZ': 3, 'rank_shifts': 2, 'n_xgens': 2, 'k': 1, 'shifts_independent': True}
PASS V2 FSC L=1: CSS code, k=1
PASS V2 D4 L=2: shifts in C_Z, commuting on C_Z, S^2=1, k=1  | {'shift_in_CZ': True, 'noncommuting_pairs': 0, 'bad_squares': 0, 'n_qubits': 36, 'dim_CZ': 12, 'rank_shifts': 11, 'n_xgens': 11, 'k': 1, 'shifts_independent': True}
PASS V2 FSC L=2: CSS code, k=1
PASS V2 D4 L=3: shifts in C_Z, commuting on C_Z, S^2=1, k=1  | {'shift_in_CZ': True, 'noncommuting_pairs': 0, 'bad_squares': 0, 'n_qubits': 72, 'dim_CZ': 27, 'rank_shifts': 26, 'n_xgens': 26, 'k': 1, 'shifts_independent': True}
PASS V2 FSC L=3: CSS code, k=1
PASS V2 D4 L=4: shifts in C_Z, commuting on C_Z, S^2=1, k=1  | {'shift_in_CZ': True, 'noncommuting_pairs': 0, 'bad_squares': 0, 'n_qubits': 120, 'dim_CZ': 48, 'rank_shifts': 47, 'n_xgens': 47, 'k': 1, 'shifts_independent': Tru

PASS V2 D4 L=5: shifts in C_Z, commuting on C_Z, S^2=1, k=1  | {'shift_in_CZ': True, 'noncommuting_pairs': 0, 'bad_squares': 0, 'n_qubits': 180, 'dim_CZ': 75, 'rank_shifts': 74, 'n_xgens': 74, 'k': 1, 'shifts_independent': True}
PASS V2 FSC L=5: CSS code, k=1
PASS V2 D4 L=6: shifts in C_Z, commuting on C_Z, S^2=1, k=1  | {'shift_in_CZ': True, 'noncommuting_pairs': 0, 'bad_squares': 0, 'n_qubits': 252, 'dim_CZ': 108, 'rank_shifts': 107, 'n_xgens': 107, 'k': 1, 'shifts_independent': True}


PASS V2 FSC L=6: CSS code, k=1
PASS V2 negative control: dressed generators without g-qubit Z checks are detected as non-commuting  | 8 non-commuting pairs


## V3. 独立実装（$L=1$、12量子ビット密ベクトル）

埋め込みの D4 ライブラリの関数を一切使わず、式(2)(3)(7)(8) を直接 4096 次元のベクトル演算として実装する。$Z$ 部分空間の計算基底を列挙し、その上で $S|\psi\rangle=|\psi\rangle$ の解空間を SVD で求め、論理 $Z$ 演算子を重み≤2 の $Z$ 型 Pauli から探索し、$U=WV$ の論理行列を求める。

In [26]:
n1 = 12; N1 = 2 ** n1; idx1 = np.arange(N1)
qq = lambda e, c: 3 * e + c        # edges: 0 bottom h(0,0), 1 top h(0,1), 2 left v(0,0), 3 right v(1,0)
bit1 = lambda i, k: (i >> (n1 - 1 - k)) & 1
def Zop(qs):
    par = np.zeros(N1, dtype=int)
    for k in qs: par ^= bit1(idx1, k)
    return lambda v: (1 - 2 * par) * v
def Xop(qs):
    m = 0
    for k in qs: m |= 1 << (n1 - 1 - k)
    return lambda v: v[idx1 ^ m]
def CZop(pairs):
    par = np.zeros(N1, dtype=int)
    for a, b in pairs: par ^= bit1(idx1, a) & bit1(idx1, b)
    return lambda v: (1 - 2 * par) * v
def compose(*fs):
    def h(v):
        for f in reversed(fs): v = f(v)
        return v
    return h
Zchecks = [[qq(e, c) for e in range(4)] for c in range(3)]
Zchecks += [[qq(2, 0)], [qq(1, 2)], [qq(0, 0), qq(0, 2)], [qq(3, 0), qq(3, 2)], [qq(0, 1)], [qq(3, 1)]]
S_rb = compose(Xop([qq(0, 0), qq(3, 0)]), Xop([qq(0, 2), qq(3, 2)]))           # vertex (1,0) on L_rb
S_g = compose(Xop([qq(2, 1), qq(1, 1)]), CZop([(qq(2, 0), qq(1, 2))]))          # vertex (0,1), v = vertex 2 of p
okZ = np.ones(N1, bool)
for zc in Zchecks:
    par = np.zeros(N1, dtype=int)
    for k in zc: par ^= bit1(idx1, k)
    okZ &= par == 0
basis = np.nonzero(okZ)[0]
Bm = np.zeros((N1, len(basis)), complex); Bm[basis, np.arange(len(basis))] = 1
rows_ = []
for S in (S_rb, S_g):
    M = np.stack([S(Bm[:, j]) for j in range(Bm.shape[1])], 1)
    assert np.allclose(M[~okZ], 0)
    rows_.append(Bm.conj().T @ M - np.eye(len(basis)))
u_, s_, vh_ = np.linalg.svd(np.concatenate(rows_))
code1 = Bm @ vh_[np.sum(s_ > 1e-9):].conj().T
record('V3a independent L=1: code dimension = 2', code1.shape[1] == 2, f'dim C_Z={len(basis)}')
P1 = code1 @ code1.conj().T
found = None
for w in (1, 2):
    for qs in itertools.combinations(range(n1), w):
        M = np.stack([Zop(qs)(code1[:, j]) for j in range(2)], 1)
        if np.allclose(P1 @ M, M):
            Mc = code1.conj().T @ M
            if not np.allclose(Mc, Mc[0, 0] * np.eye(2)):
                found = (qs, Mc); break
    if found: break
ev, evec = np.linalg.eigh(found[1]); logical1 = code1 @ evec[:, ::-1]
def CNOT1(c, t):
    return lambda v: np.where(bit1(idx1, c) == 1, v[idx1 ^ (1 << (n1 - 1 - t))], v)
V1 = compose(*[CNOT1(qq(e, c), qq(e, 1)) for e in range(4) for c in (0, 2)])
def phase1(terms):
    ph = np.zeros(N1)
    for c, qs in terms:
        m = np.ones(N1, dtype=int)
        for k in qs: m &= bit1(idx1, k)
        ph = ph + c * m
    return lambda v: np.exp(1j * np.pi / 4 * ph) * v
W1 = phase1([(2, (qq(0, 0), qq(3, 2))), (-2, (qq(2, 0), qq(1, 2)))] + [(-1, (qq(e, 0),)) for e in (0, 3)])
U1 = compose(W1, V1)
UL = np.stack([U1(logical1[:, j]) for j in range(2)], 1)
Um = logical1.conj().T @ UL; Um = Um / Um[0, 0]
record('V3b independent L=1: U preserves the code space', np.allclose(P1 @ UL, UL))
record('V3c independent L=1: logical action of U = T^dag (Z-basis defined by the trivial gauge field)',
       np.allclose(Um, np.diag([1, np.exp(-1j * np.pi / 4)])), np.round(Um, 4))

PASS V3a independent L=1: code dimension = 2  | dim C_Z=8


PASS V3b independent L=1: U preserves the code space
PASS V3c independent L=1: logical action of U = T^dag (Z-basis defined by the trivial gauge field)  | [[1.    +0.j     0.    +0.j    ]  [0.    +0.j     0.7071-0.7071j]]


## V4. 論理演算子と距離（MILP を全探索で照合）
$d_Z$：$Su=0$, $t_{\log}\cdot u=1$ を満たす最小重み $u$。$d_X^{\rm shift}$：$H_Z t=0$, $z_{\log}\cdot t=1$ を満たす最小重み $t$。frozen 量子ビットを含む最小解は無い（frozen 上の $Z$ は安定化子、$X$ は検出される）ので frozen を除いて全探索する。

In [27]:
def brute_min(Hrows, lvec, cols, wmax):
    Hc = Hrows[:, cols].astype(np.int64); lc = lvec[cols].astype(np.int64)
    for w in range(1, wmax + 1):
        for sup in itertools.combinations(range(len(cols)), w):
            sup = list(sup)
            if (lc[sup].sum() % 2 == 1) and not (Hc[:, sup].sum(axis=1) % 2).any():
                return w
    return None
for L, kind in [(2, 'Z'), (3, 'Z'), (2, 'X')]:
    lat = D4Lattice(L); fz = set()
    for lab, rows in lat.z_checks():
        if lab[0] == 'bdZ' and len(rows) == 1: fz |= rows
    for name, cd in [('FSC', CodeData(lat, (R, B), dressed=False)), ('D4', CodeData(lat))]:
        cols = [c for c in cd.cols if c not in fz]
        dz, dx, _, _ = distances(cd)
        if kind == 'Z':
            bf = brute_min(cd.S, cd.t_log, cols, dz)
            record(f'V4 {name} L={L}: brute-force d_Z = MILP d_Z = L', bf == dz == L, (bf, dz))
        else:
            if len(cols) > 50:
                continue
            bf = brute_min(cd.HZ, cd.z_log, cols, dx)
            record(f'V4 {name} L={L}: brute-force X-shift distance = MILP = L+3', bf == dx == L + 3, (bf, dx))

PASS V4 FSC L=2: brute-force d_Z = MILP d_Z = L  | (2, 2)
PASS V4 D4 L=2: brute-force d_Z = MILP d_Z = L  | (2, 2)
PASS V4 FSC L=3: brute-force d_Z = MILP d_Z = L  | (3, 3)


PASS V4 D4 L=3: brute-force d_Z = MILP d_Z = L  | (3, 3)
PASS V4 FSC L=2: brute-force X-shift distance = MILP = L+3  | (5, 5)


PASS V4 D4 L=2: brute-force X-shift distance = MILP = L+3  | (5, 5)


## V5. Knill–Laflamme 条件（$L=2$）
$\langle i|E|j\rangle = c_E\delta_{ij}$ を、論理基底 $|0_L\rangle,|1_L\rangle$（各 $2^{11}$ 項の疎ベクトル）と全ての weight≤2 Pauli $E$ について直接検査する（約1分）。D4 相の真の距離が $L=2$ で 2 であることを確認する（$L\ge3$ は**未検証**）。

In [28]:
t0 = time.time()
lat2 = D4Lattice(2); d42 = CodeData(lat2)
kb = [coset_state(d42, set()), coset_state(d42, set(np.nonzero(d42.t_log)[0]))]
def apply_pauli(st, xs, zs):
    xs = frozenset(xs); out = {}
    for x, v in st.a.items():
        out[x ^ xs] = v * (-1) ** len(x & zs)
    return SparseState(out)
viol = {}
for w in (1, 2):
    viol[w] = 0
    for qs in itertools.combinations(range(lat2.n), w):
        for ps in itertools.product('XYZ', repeat=w):
            xs = [a for a, p in zip(qs, ps) if p in 'XY']; zs = {a for a, p in zip(qs, ps) if p in 'YZ'}
            M = np.array([[kb[i].overlap(apply_pauli(kb[j], xs, zs)) for j in range(2)] for i in range(2)])
            if abs(M[0, 1]) > 1e-9 or abs(M[1, 0]) > 1e-9 or abs(M[0, 0] - M[1, 1]) > 1e-9:
                viol[w] += 1
record('V5 D4 L=2 Knill-Laflamme: all weight-1 Paulis correctable-detectable, some weight-2 logical (distance = 2)',
       viol[1] == 0 and viol[2] > 0, f'violations {viol}, {time.time() - t0:.0f}s')

PASS V5 D4 L=2 Knill-Laflamme: all weight-1 Paulis correctable-detectable, some weight-2 logical (distance = 2)  | violations {1: 0, 2: 2}, 104s


## V5b. D4 相の Knill–Laflamme 距離 $=L$ の厳密証明（$L\le 5$、`FULL_KL=True` で $L=6$）

$E=X^tZ^u$ について：(i) $t\notin C_Z$ なら $E$ は符号空間を直交空間へ写す（検出可能）。(ii) $t$ が論理類なら $|t|\ge L+3$（V4）。(iii) $t\in\mathrm{span}(S)$ なら $X^t=S_cF_c$（$S_c$ は安定化子の積、$F_c=(-1)^{f_c}$ は対角）なので $PEP=PF_cZ^uP$。$F_c$ を $Z$ 型 Pauli $Z^v$（$v\subseteq Q_f=\mathrm{supp}f_c$）に展開すると、KL 条件が破れるには $v+u$ が論理 $Z$ でなければならない。よって
$$|E|\ \ge\ |t| + \min_{\ell:\ \text{logical }Z}|\ell\setminus(Q_f\cup t)|=:LB_t .$$
重み $\le L-1$ の全ての $t\in\mathrm{span}(S)$ を MILP（no-good cut 付き）で**厳密に列挙**し、$\min_t LB_t\ge L$ を確認する。$t=0$ は $d_Z=L$ で扱える。重み $L$ の論理 $Z$ があるので距離はちょうど $L$。

In [29]:
FULL_KL = False
for L in range(3, 7 if FULL_KL else 6):
    t0 = time.time()
    P_ = D4Protocol(L)
    lb, ts, det, stop = kl_distance_lower_bound(P_.d4, L - 1, P_.frozen)
    dz, dx, _, _ = distances(P_.d4)
    # stop is 'infeasible' or 'min weight > wmax': the enumeration ended because the solver
    # proved there is no further t (any other solver status raises inside _milp_binary)
    ok = stop in ('infeasible', 'min weight > wmax') and (lb is None or lb >= L) and dz == L and dx >= L
    record(f'V5b L={L}: every Pauli error of weight <= L-1 satisfies Knill-Laflamme -> D4-phase distance = L',
           ok, f'{len(ts)} stabilizer shifts of weight <= {L-1} enumerated (stop: {stop}); min LB_t = {lb}; d_Z = {dz}; X-shift = {dx}; {time.time()-t0:.0f}s')

PASS V5b L=3: every Pauli error of weight <= L-1 satisfies Knill-Laflamme -> D4-phase distance = L  | 1 stabilizer shifts of weight <= 2 enumerated (stop: infeasible); min LB_t = 5; d_Z = 3; X-shift = 6; 0s


PASS V5b L=4: every Pauli error of weight <= L-1 satisfies Knill-Laflamme -> D4-phase distance = L  | 15 stabilizer shifts of weight <= 3 enumerated (stop: infeasible); min LB_t = 5; d_Z = 4; X-shift = 7; 3s


PASS V5b L=5: every Pauli error of weight <= L-1 satisfies Knill-Laflamme -> D4-phase distance = L  | 84 stabilizer shifts of weight <= 4 enumerated (stop: infeasible); min LB_t = 6; d_Z = 5; X-shift = 8; 48s


## V6. transversal $U=WV$（式(5)(8)）

各生成元 $S$ について、[KZH] SM の式（permuting stabilizers）が予言する像 $S'$（$S^r\to S^rS^g$, $S^g\to S^g$, $S^b\to S^gS^b$, $\mathcal L_{rb}$ 上の $S^rS^b\to S^rS^b$）と $USU^\dagger$ が $C_Z$ 上で一致することを、**位相差が依存する量子ビットへの $C_Z$ の射影を厳密に全列挙**して確認する。
負の対照：(i) 境界の $T^\dagger$ を除く、(ii) CS と CS$^\dagger$ を入れ替える、(iii) 境界を $T$ にする、(iv) 両方 CS にする → 失敗するか論理作用が変わるはず。

In [30]:
def check_U(L, cs=(1, -1), bt=-1):
    lat = D4Lattice(L); cd = CodeData(lat); Wop = W_operator(lat, cs, bt)
    fails, mx = [], 0
    for lab, op in cd.xgens:
        ok, n = check_conjugation(cd, Wop, op, U_target(lat, lab))
        mx = max(mx, n)
        if not ok: fails.append(lab)
    return fails, mx, logical_phases(cd, Wop), cd
for L in range(1, 7):
    fails, mx, ph, cd = check_U(L)
    # V maps C_Z into C_Z
    Bz = CZ_basis(cd)
    VB = np.array([cd.lat.vec(V_map(cd.lat, frozenset(np.nonzero(b)[0]))) for b in Bz])
    vz = not ((cd.HZ.astype(int) @ VB.T.astype(int)) % 2).any()
    record(f'V6 L={L}: U S U^dag = S\' on C_Z for all {len(cd.xgens)} generators; V(C_Z)=C_Z; logical phases (pi/4 units) = [0,7] (T^dag)',
           not fails and vz and ph == [0, 7], f'max local configs {mx}, phases {ph}')
for name, cs, bt in [('no boundary T^dag', (1, -1), 0), ('CS<->CS^dag swapped', (-1, 1), -1),
                     ('boundary T instead of T^dag', (1, -1), 1), ('CS,CS (no dagger)', (1, 1), -1)]:
    fails, mx, ph, _ = check_U(4, cs, bt)
    record(f'V6 negative control L=4 [{name}] is rejected or changes the logical action', bool(fails) or ph != [0, 7],
           f'{len(fails)} failing generators, phases {ph}')
fails, mx, ph, _ = check_U(4, (-1, 1), 1)
record('V6 conjugate convention (CS^dag,CS, boundary T) gives a valid logical T (consistency)', not fails and ph == [0, 1], ph)

PASS V6 L=1: U S U^dag = S' on C_Z for all 2 generators; V(C_Z)=C_Z; logical phases (pi/4 units) = [0,7] (T^dag)  | max local configs 4, phases [0, 7]
PASS V6 L=2: U S U^dag = S' on C_Z for all 11 generators; V(C_Z)=C_Z; logical phases (pi/4 units) = [0,7] (T^dag)  | max local configs 1024, phases [0, 7]


PASS V6 L=3: U S U^dag = S' on C_Z for all 26 generators; V(C_Z)=C_Z; logical phases (pi/4 units) = [0,7] (T^dag)  | max local configs 4096, phases [0, 7]


PASS V6 L=4: U S U^dag = S' on C_Z for all 47 generators; V(C_Z)=C_Z; logical phases (pi/4 units) = [0,7] (T^dag)  | max local configs 4096, phases [0, 7]


PASS V6 L=5: U S U^dag = S' on C_Z for all 74 generators; V(C_Z)=C_Z; logical phases (pi/4 units) = [0,7] (T^dag)  | max local configs 4096, phases [0, 7]


PASS V6 L=6: U S U^dag = S' on C_Z for all 107 generators; V(C_Z)=C_Z; logical phases (pi/4 units) = [0,7] (T^dag)  | max local configs 4096, phases [0, 7]


PASS V6 negative control L=4 [no boundary T^dag] is rejected or changes the logical action  | 7 failing generators, phases [0, 0]


PASS V6 negative control L=4 [CS<->CS^dag swapped] is rejected or changes the logical action  | 7 failing generators, phases [0, 7]


PASS V6 negative control L=4 [boundary T instead of T^dag] is rejected or changes the logical action  | 7 failing generators, phases [0, 1]


PASS V6 negative control L=4 [CS,CS (no dagger)] is rejected or changes the logical action  | 24 failing generators, phases [0, 7]


PASS V6 conjugate convention (CS^dag,CS, boundary T) gives a valid logical T (consistency)  | [0, 1]


## V7. プロトコル全体の厳密シミュレーション（$L=1,2$）

[KZH] の手順を疎状態ベクトル（計算基底の辞書）でそのまま実行する。
1. FSC の論理状態 $\alpha|0_L\rangle+\beta|1_L\rangle$（ランダム）、$g$ は $|0\rangle$。全 $Z$ 安定化子が $+1$ であることを確認。
2. D4 の全 $X$ 型生成元を射影測定（ボルン則で分岐をサンプル）。2回目の測定で結果が再現すること（ノイズなしで1ラウンドで十分）を確認。
3. $U=WV$。
4. 全 $Z^g_e$ を測定し、$\delta\kappa=a_g$ を解いて $\prod_{v\in\kappa}S^g_{X;v}$ を適用。
5. $m_v=1$ の各頂点に $\mathcal L_{rb}$ から $Z^rZ^b$ 文字列（Pauli フレーム補正）。
6. 目標状態 $\alpha|0_L\rangle+e^{-i\pi/4}\beta|1_L\rangle$ との忠実度。

負の対照：補正5を外す／補正4の CZ 部分を外す（$X^g$ のみ）／境界 $T^\dagger$ を外す。

In [31]:
def run_protocol(L, alpha, beta, rng, skip_pauli=False, sg_x_only=False, W_bt=-1):
    lat = D4Lattice(L); fsc = CodeData(lat, (R, B), dressed=False); d4 = CodeData(lat)
    zero = coset_state(fsc, set()); one = coset_state(fsc, set(np.nonzero(fsc.t_log)[0]))
    psi = SparseState({k: alpha * v for k, v in zero.a.items()})
    for k, v in one.a.items(): psi.a[k] = psi.a.get(k, 0) + beta * v
    for lab, rows in lat.z_checks():
        assert abs(psi.expect(PSOp(lin=rows)) - 1) < 1e-9
    m = {}
    for lab, op in d4.xgens:
        bit, psi, p = psi.measure_op(op, rng); m[lab] = bit
    repeat_ok = all(abs(psi.expect(op) - (-1) ** m[lab]) < 1e-9 for lab, op in d4.xgens)
    zero_ok = all(abs(psi.expect(PSOp(lin=rows)) - 1) < 1e-9 for lab, rows in lat.z_checks())
    psi = psi.apply_perm(lambda x: V_map(lat, x)).apply_diag(W_operator(lat, (1, -1), W_bt))
    ag = set()
    for e in range(lat.E):
        bit, psi, p = psi.measure_z(lat.q(e, G), rng)
        if bit: ag.add(e)
    ggens = [(lab, op) for lab, op in d4.xgens if lab[2] == 'g']
    Sg = np.array([lat.vec(op.s) for _, op in ggens])
    kappa = f2_solve(Sg, lat.vec({lat.q(e, G) for e in ag}))
    for i in np.nonzero(kappa)[0]:
        op = ggens[i][1]
        psi = psi.apply_psop(PSOp(s=op.s) if sg_x_only else op)
    gclean = all(all(lat.q(e, G) not in x for e in range(lat.E)) for x in psi.a)
    if not skip_pauli:
        for lab, op in ggens:
            if m[lab]:
                x0, y0 = lab[1]
                qs = {lat.q(lat.eidx[('v', x0, y)], c) for y in range(y0) for c in (R, B)}
                psi = psi.apply_diag(DiagPhase([(4, (q,)) for q in qs]))
    target = SparseState({k: alpha * v for k, v in zero.a.items()})
    for k, v in one.a.items(): target.a[k] = target.a.get(k, 0) + beta * OMEGA ** 7 * v
    F = abs(target.overlap(psi)) ** 2 / (target.norm2() * psi.norm2())
    mg = tuple(m[lab] for lab, _ in ggens)
    return dict(F=F, repeat_ok=repeat_ok, zero_ok=zero_ok, g_cleared=gclean, m_g=mg,
                m_other_nontrivial=any(m[lab] for lab in m if lab[2] != 'g'), kappa_weight=int(kappa.sum()),
                n_g=len(ggens))
def rand_state(rng):
    a, b = rng.normal(size=2) + 1j * rng.normal(size=2); nrm = np.hypot(abs(a), abs(b)); return a / nrm, b / nrm
stats = {}
for L, ntr in [(1, 60), (2, 200 if not QUICK else 40)]:
    t0 = time.time(); res = [run_protocol(L, *rand_state(rng), rng) for _ in range(ntr)]
    Fmin = min(r['F'] for r in res)
    record(f'V7 L={L}: exact protocol, {ntr} random inputs/branches -> fidelity 1 with T^dag action',
           Fmin > 1 - 1e-9 and all(r['repeat_ok'] and r['zero_ok'] and r['g_cleared'] for r in res)
           and not any(r['m_other_nontrivial'] for r in res), f'min F = {Fmin:.12f}, {time.time() - t0:.0f}s')
    stats[L] = res
# basis inputs (logical information preserved for |0>, |1>, |+>, |+i>)
for name, (a, b) in {'|0>': (1, 0), '|1>': (0, 1), '|+>': (2 ** -.5, 2 ** -.5), '|+i>': (2 ** -.5, 1j * 2 ** -.5)}.items():
    Fs = [run_protocol(2, a, b, rng)['F'] for _ in range(10)]
    record(f'V7 L=2 input {name}: output = T^dag {name}', min(Fs) > 1 - 1e-9, f'min F={min(Fs):.12f}')
res2 = stats[2]; ng = res2[0]['n_g']
cm = Counter(r['m_g'] for r in res2); ck = Counter(r['kappa_weight'] for r in res2)
from scipy.stats import chisquare
p_m = chisquare([cm.get(t, 0) for t in itertools.product((0, 1), repeat=ng)]).pvalue
record(f'V7 L=2: S^g outcomes m are random (all {2**ng} patterns occur; chi^2 test vs uniform p={p_m:.2f})',
       len(cm) == 2 ** ng and p_m > 1e-3, dict(sorted(cm.items())))
mean_k = np.mean([r['kappa_weight'] for r in res2])
record('V7 L=2: switch-out correction |kappa| has mean ~ n_g/2 (supports expected CZ count = worst/2)',
       abs(mean_k - ng / 2) < 0.35, f'mean |kappa| = {mean_k:.2f}, n_g = {ng}, hist {dict(sorted(ck.items()))}')
negs = {'without Z^rZ^b Pauli-frame correction': dict(skip_pauli=True),
        'S^g correction without its CZ part': dict(sg_x_only=True),
        'W without boundary T^dag': dict(W_bt=0)}
for name, kw in negs.items():
    Fs = [run_protocol(2, *rand_state(rng), rng, **kw)['F'] for _ in range(60)]
    record(f'V7 negative control [{name}] fails for some branches', min(Fs) < 1 - 1e-6, f'min F = {min(Fs):.4f}')

PASS V7 L=1: exact protocol, 60 random inputs/branches -> fidelity 1 with T^dag action  | min F = 1.000000000000, 0s


PASS V7 L=2: exact protocol, 200 random inputs/branches -> fidelity 1 with T^dag action  | min F = 1.000000000000, 117s


PASS V7 L=2 input |0>: output = T^dag |0>  | min F=1.000000000000


PASS V7 L=2 input |1>: output = T^dag |1>  | min F=1.000000000000


PASS V7 L=2 input |+>: output = T^dag |+>  | min F=1.000000000000


PASS V7 L=2 input |+i>: output = T^dag |+i>  | min F=1.000000000000


PASS V7 L=2: S^g outcomes m are random (all 16 patterns occur; chi^2 test vs uniform p=0.58)  | {(0, 0, 0, 0): 15, (0, 0, 0, 1): 8, (0, 0, 1, 0): 8, (0, 0, 1, 1): 17, (0, 1, 0, 0): 12, (0, 1, 0, 1): 15, (0, 1, 1, 0): 17, (0, 1, 1, 1): 10, (1, 0, 0, 0): 10, (1, 0, 0, 1): 16, (1, 0, 1, 0): 12, (1, 0, 1, 1): 10, (1, 1, 0, 0): 17, (1, 1, 0, 1): 8, (1, 1, 1, 0): 13, (1, 1, 1, 1): 12}
PASS V7 L=2: switch-out correction |kappa| has mean ~ n_g/2 (supports expected CZ count = worst/2)  | mean |kappa| = 2.19, n_g = 4, hist {0: 8, 1: 40, 2: 76, 3: 57, 4: 19}


PASS V7 negative control [without Z^rZ^b Pauli-frame correction] fails for some branches  | min F = 0.0000


PASS V7 negative control [S^g correction without its CZ part] fails for some branches  | min F = 0.0625


PASS V7 negative control [W without boundary T^dag] fails for some branches  | min F = 0.5011


## V7b. プロトコル（回路）と位相公式 $F(a)$ の照合（$L=3,4$）

V8 の局所証明は配置ごとの位相の式 $F(a)$ に基づく。その式自体を、$L=1,2$ の符号状態シミュレーション（V7）とは別に、**実際の測定・$U$・補正の回路**を単一の配置入力 $|a\rangle|0\rangle_g$ に作用させて $L=3,4$ で照合する（ノイズなし最小版：$S^g_X$ のみ測定）。同じ測定分岐（$m$ と $Z^g$ の結果を固定）で 2 つの配置 $a_1,a_2$（異なる論理剰余類を含む）の出力振幅比が $\omega^{F(a_2)-F(a_1)}$ に一致すること＝回路が配置ごとに $F$ の位相を与えること、を確認する。プロトコルは線形なので、これは符号状態に対する作用を決める。

In [32]:
def protocol_on_config(lat, d4, a, rng, forced_m=None, forced_z=None):
    psi = SparseState({frozenset(a): 1.0})
    ggens = [(lab, op) for lab, op in d4.xgens if lab[2] == 'g']
    m = []
    for i, (lab, op) in enumerate(ggens):
        bit, psi, p = psi.measure_op(op, rng, forced=None if forced_m is None else forced_m[i]); m.append(bit)
    psi = psi.apply_perm(lambda x: V_map(lat, x)).apply_diag(W_operator(lat))
    zb = []
    for j, e in enumerate(range(lat.E)):
        bit, psi, p = psi.measure_z(lat.q(e, G), rng, forced=None if forced_z is None else forced_z[j]); zb.append(bit)
        if p < 1e-12: return None, m, zb
    ag = {lat.q(e, G) for e, b in zip(range(lat.E), zb) if b}
    kappa = f2_solve(np.array([lat.vec(op.s) for _, op in ggens]), lat.vec(ag))
    for i in np.nonzero(kappa)[0]:
        psi = psi.apply_psop(ggens[i][1])
    for (lab, op), bit in zip(ggens, m):
        if bit:
            x0, y0 = lab[1]
            psi = psi.apply_diag(DiagPhase([(4, (lat.q(lat.eidx[('v', x0, y)], c),)) for y in range(y0) for c in (R, B)]))
    assert set(psi.a) == {frozenset(a)}, 'output not proportional to the input configuration'
    return psi.a[frozenset(a)], m, zb
for L, ntr in [(3, 6), (4, 2 if QUICK else 4)]:
    t0 = time.time()
    lat = D4Lattice(L); d4 = CodeData(lat); fsc = CodeData(lat, (R, B), dressed=False)
    gverts = [lab[1] for lab, op in d4.xgens if lab[2] == 'g']; Wop = W_operator(lat)
    def rand_cfg(coset):
        x = set()
        for i in np.nonzero(rng.integers(0, 2, len(fsc.xgens)))[0]: x ^= set(fsc.xgens[i][1].s)
        if coset: x ^= set(np.nonzero(fsc.t_log)[0])
        return frozenset(x)
    ok, nb = True, 0
    for _ in range(ntr):
        a1 = rand_cfg(0)
        A1, m, zb = protocol_on_config(lat, d4, a1, rng)
        for a2 in [rand_cfg(0), rand_cfg(1), rand_cfg(1)]:
            A2, _, _ = protocol_on_config(lat, d4, a2, rng, forced_m=m, forced_z=zb)
            if A2 is None: continue
            nb += 1
            pred = OMEGA ** ((protocol_phase(lat, Wop, a2, gverts) - protocol_phase(lat, Wop, a1, gverts)) % 8)
            ok &= abs(A2 / A1 - pred) < 1e-9
    record(f'V7b L={L}: protocol circuit on configuration inputs reproduces the phase formula F(a) (incl. T^dag between cosets)',
           ok and nb > 0, f'{nb} same-branch pairs, {time.time()-t0:.0f}s')

PASS V7b L=3: protocol circuit on configuration inputs reproduces the phase formula F(a) (incl. T^dag between cosets)  | 18 same-branch pairs, 0s


PASS V7b L=4: protocol circuit on configuration inputs reproduces the phase formula F(a) (incl. T^dag between cosets)  | 12 same-branch pairs, 56s


## V8. 大きな $L$ でのプロトコルの正しさ（局所厳密証明）

V7 の手順を解析すると、FSC の配置 $a=(a_r,a_b)$ が受け取る位相は
$$F(a) = w(a) + 4\sum_{v} c'_v(a)\, f^g_v(a)\pmod 8,\qquad \delta c' = a_r + a_b,\; c'|_{\mathcal L_{rb}}=0$$
（$w$ は $W$ の位相指数、$f^g_v$ は $S^g_{X;v}$ の CZ$^{r,b}$ 部分）となる（$m$ に依存する部分は Pauli 補正で消え、$\kappa$ に依存する部分は $S^g$ 補正で消える）。この式は V7 の厳密シミュレーションで $L\le2$ について裏付けられている。
プロトコルが論理 $T^\dagger$ を実装する ⇔ $F$ が論理剰余類上で一定で $F(1)-F(0)=7$。
FSC の各生成元シフト $t$ について $F(a+t)=F(a)$ を、$F(a+t)-F(a)$ が依存する量子ビット（と $c'_{w_0}(a)$ という $\mathbb F_2$ 線形汎関数）への $C_{\rm FSC}$ の**厳密な射影**の全要素で確認する。

In [33]:
for L in range(1, L_MAX_INVARIANCE + 1):
    t0 = time.time()
    lat = D4Lattice(L); fsc = CodeData(lat, (R, B), dressed=False); d4 = CodeData(lat)
    gverts = [lab[1] for lab, op in d4.xgens if lab[2] == 'g']; Wop = W_operator(lat)
    fails, mx = 0, 0
    for lab, op in fsc.xgens:
        ok, nloc = check_protocol_invariance(lat, fsc, Wop, gverts, op); fails += (not ok); mx = max(mx, nloc)
    F0 = protocol_phase(lat, Wop, frozenset(), gverts)
    F1 = protocol_phase(lat, Wop, frozenset(np.nonzero(fsc.t_log)[0]), gverts)
    record(f'V8 L={L}: F invariant under all {len(fsc.xgens)} FSC generators; F(1)-F(0)=7 (T^dag)',
           fails == 0 and (F1 - F0) % 8 == 7, f'max local configs {mx}, F0={F0}, F1={F1}, {time.time() - t0:.1f}s')

PASS V8 L=1: F invariant under all 1 FSC generators; F(1)-F(0)=7 (T^dag)  | max local configs 4, F0=0, F1=7, 0.0s
PASS V8 L=2: F invariant under all 7 FSC generators; F(1)-F(0)=7 (T^dag)  | max local configs 64, F0=0, F1=7, 0.0s


PASS V8 L=3: F invariant under all 17 FSC generators; F(1)-F(0)=7 (T^dag)  | max local configs 2048, F0=0, F1=7, 0.2s


PASS V8 L=4: F invariant under all 31 FSC generators; F(1)-F(0)=7 (T^dag)  | max local configs 4096, F0=0, F1=7, 0.7s


PASS V8 L=5: F invariant under all 49 FSC generators; F(1)-F(0)=7 (T^dag)  | max local configs 16384, F0=0, F1=7, 2.1s


PASS V8 L=6: F invariant under all 71 FSC generators; F(1)-F(0)=7 (T^dag)  | max local configs 16384, F0=0, F1=7, 4.6s


PASS V8 L=7: F invariant under all 97 FSC generators; F(1)-F(0)=7 (T^dag)  | max local configs 16384, F0=0, F1=7, 7.8s


PASS V8 L=8: F invariant under all 127 FSC generators; F(1)-F(0)=7 (T^dag)  | max local configs 16384, F0=0, F1=7, 13.5s


## V9. 測定ガジェット（制御-$S$ ＋ $X$ 測定）の正しさ

$L=3$ の全 $X$ 型生成元（frozen を除かない版：最も多くの CCZ を含む）について、ガジェット回路をその台の上の密ベクトルで実行し、補助を $|\pm\rangle$ に射影した後のデータ状態が $(1\pm S)|\psi\rangle/2$ と一致することを確認する（$|\psi\rangle$ はランダム）。frozen 量子ビットを除いた版は、frozen を $|0\rangle$ とした状態で同じ検査を行う。

In [34]:
def gadget_dense_check(P, lab, rng, frozen_zero=False):
    gad = P.x_gadget(lab, True)
    op = PSOp()
    for ch in lab[2]:
        op = op @ P.lat.SX(lab[1], CNAME.index(ch))
    anc = gad[0][1]
    qs = sorted((gadget_support(gad) | op.support()) - {anc}) + [anc]
    k = len(qs); pos = {q: i for i, q in enumerate(qs)}; idx = np.arange(2 ** k)
    b = lambda q: (idx >> (k - 1 - pos[q])) & 1
    psi = rng.normal(size=2 ** k) + 1j * rng.normal(size=2 ** k)
    psi = np.where(b(anc) == 0, psi, 0)
    if frozen_zero:
        for q in qs:
            if q in P.frozen: psi = np.where(b(q) == 0, psi, 0)
    psi /= np.linalg.norm(psi)
    st = (psi + psi[idx ^ (1 << (k - 1 - pos[anc]))]) / np.sqrt(2)          # ancilla |0> -> |+>
    for g in gad[1:-1]:
        if g[0] == 'CNOT':
            st = np.where(b(g[1]) == 1, st[idx ^ (1 << (k - 1 - pos[g[2]]))], st)
        elif g[0] == 'CCZ':
            st = np.where(b(g[1]) & b(g[2]) & b(g[3]), -st, st)
        else:
            raise ValueError(g)
    Spsi = np.zeros_like(psi)
    for i in np.nonzero(np.abs(psi) > 0)[0]:
        x = frozenset(q for q in qs if q != anc and (i >> (k - 1 - pos[q])) & 1)
        y, sb = op.apply(x)
        j = sum(1 << (k - 1 - pos[q]) for q in y)
        Spsi[j] += (-1) ** sb * psi[i]
    ok = True
    for sgn in (1, -1):
        a0 = np.where(b(anc) == 0, st, 0); a1 = np.where(b(anc) == 1, st, 0)
        post = (a0 + sgn * a1[idx ^ (1 << (k - 1 - pos[anc]))]) / np.sqrt(2)
        ok &= np.allclose(post, (psi + sgn * Spsi) / 2)
    return ok
Pu = D4Protocol(3, prune_frozen=False); Pp = D4Protocol(3, prune_frozen=True)
okU = all(gadget_dense_check(Pu, lab, rng) for lab, _ in Pu.d4.xgens)
okP = all(gadget_dense_check(Pp, lab, rng, frozen_zero=True) for lab, _ in Pp.d4.xgens)
record('V9 L=3: every X-type gadget (unpruned) implements the projective measurement of its Clifford stabilizer', okU)
record('V9 L=3: pruned gadgets (frozen qubits removed) agree when frozen qubits are |0>', okP)

PASS V9 L=3: every X-type gadget (unpruned) implements the projective measurement of its Clifford stabilizer
PASS V9 L=3: pruned gadgets (frozen qubits removed) agree when frozen qubits are |0>


## V10. スケジュールの妥当性（独立な行列計算）

可換クラス・スケジュールの上界は「同じクラスの異なるガジェットのゲートは全て可換」という条件に依存する（その場合、同時実行は逐次測定と等価）。役割による判定とは独立に、共有量子ビットを持つゲート対の行列（≤6量子ビット）を作って交換子を計算する。$U$ のゲート同士の可換性も同様に確認する（$U$ を1層に詰めてよい根拠）。

In [35]:
GMAT = {'CNOT': np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], complex),
        'CZ': np.diag([1, 1, 1, -1]).astype(complex), 'CS': np.diag([1, 1, 1, 1j]), 'CSDG': np.diag([1, 1, 1, -1j]),
        'CCZ': np.diag([1] * 7 + [-1]).astype(complex), 'T': np.diag([1, np.exp(1j * np.pi / 4)]),
        'TDG': np.diag([1, np.exp(-1j * np.pi / 4)])}
def embed(g, qs):
    k = len(qs); pos = {q: i for i, q in enumerate(qs)}; G_ = GMAT[g[0]]; a = len(g) - 1
    M = np.zeros((2 ** k, 2 ** k), complex)
    for col in range(2 ** k):
        bits = [(col >> (k - 1 - i)) & 1 for i in range(k)]
        sub = sum(bits[pos[q]] << (a - 1 - t) for t, q in enumerate(g[1:]))
        for out in range(2 ** a):
            amp = G_[out, sub]
            if amp == 0: continue
            nb = bits[:]
            for t, q in enumerate(g[1:]): nb[pos[q]] = (out >> (a - 1 - t)) & 1
            M[sum(bb << (k - 1 - i) for i, bb in enumerate(nb)), col] += amp
    return M
_cache = {}
def commute(g1, g2):
    key = (g1, g2)
    if key not in _cache:
        qs = sorted(set(g1[1:]) | set(g2[1:])); A, Bm_ = embed(g1, qs), embed(g2, qs)
        _cache[key] = np.allclose(A @ Bm_, Bm_ @ A)
    return _cache[key]
P3 = D4Protocol(3)
allok, npairs = True, 0
for code_ in ['D4', 'FSC']:
    for cls_ in commuting_classes(P3.round_gadgets(code_)):
        ents = [[g for g in gad if g[0] in GMAT] for gad in cls_]
        byq = defaultdict(list)
        for gi, gl in enumerate(ents):
            for g in gl:
                for q in g[1:]: byq[q].append((gi, g))
        for q, lst in byq.items():
            for (i1, g1), (i2, g2) in itertools.combinations(lst, 2):
                if i1 != i2:
                    npairs += 1
                    if not commute(g1, g2): allok = False
record('V10 L=3: all gates of different gadgets within each commuting class commute (matrix check)', allok, f'{npairs} gate pairs')
Ug = P3.U_gates(); okUc = True
for g1, g2 in itertools.combinations(Ug, 2):
    if set(g1[1:]) & set(g2[1:]) and not commute(g1, g2): okUc = False
record('V10 L=3: all gates of U = W V mutually commute (U can be layered freely)', okUc, f'{len(Ug)} gates')
# Clifford+T decompositions used in the 'clifford_t' gate-set model
def circuit_matrix(gates, k):
    M = np.eye(2 ** k, dtype=complex)
    for g in gates: M = embed(g, list(range(k))) @ M
    return M
record('V10 CCZ = 7-T decomposition (matrix)', np.allclose(circuit_matrix(ccz_to_clifford_t(0, 1, 2), 3), GMAT['CCZ']))
record('V10 CS = 3-T decomposition (matrix)', np.allclose(circuit_matrix(cs_to_clifford_t(0, 1), 2), GMAT['CS']))
record('V10 CS^dag decomposition (matrix)', np.allclose(circuit_matrix(csdg_to_clifford_t(0, 1), 2), GMAT['CSDG']))

PASS V10 L=3: all gates of different gadgets within each commuting class commute (matrix check)  | 184 gate pairs
PASS V10 L=3: all gates of U = W V mutually commute (U can be layered freely)  | 49 gates
PASS V10 CCZ = 7-T decomposition (matrix)
PASS V10 CS = 3-T decomposition (matrix)
PASS V10 CS^dag decomposition (matrix)


## V11. 資源計数の独立照合（幾何からの直接公式）
格子の幾何から手で導いた式と、ガジェット生成コードの計数を比較する。
* データ（frozen 除く）：辺数 $2L(L+1)$ × 3色 − frozen（$\mathcal L_r$ の $r$: $L$、$\mathcal L_b$ の $b$: $L$、$\mathcal L_{rb}$ の $g$: $2L$）$=6L^2+2L$。
* $X$ 型生成元：バルク $3(L-1)^2$、$\mathcal L_r$・$\mathcal L_b$ 各 $2(L-1)$、左上角1、$\mathcal L_{rb}$ $2L-1$ → 合計 $3L^2-2L+1$… などを直接列挙と比較。
* CCZ／ラウンド（frozen 除去後）：CZ 対のうち両端が非 frozen のもの。

In [36]:
ok = True; rows = []
for L in range(2, 10):
    P = D4Protocol(L); qs = P.qubit_sets()
    E = 2 * L * (L + 1)
    data_formula = 3 * E - (L + L + 2 * L)
    xg_formula = 3 * (L - 1) ** 2 + 2 * (L - 1) + 2 * (L - 1) + 1 + (2 * L - 1)
    zc_formula = 3 * L * L + 2 * L        # plaquettes (3 colours) + Z^rZ^b on the 2L L_rb edges
    ccz_direct = sum(1 for lab, _ in P.d4.xgens for ch in lab[2]
                     for p in P.lat.cz_pairs(lab[1], CNAME.index(ch)) if p[0] not in P.frozen and p[1] not in P.frozen)
    ccz_code = sum(1 for gd in P.round_gadgets('D4') for g in gd if g[0] == 'CCZ')
    r = dict(L=L, data=len(qs['d4_data']), data_formula=data_formula, xgens=len(P.d4.xgens), xg_formula=xg_formula,
             anc=len(qs['d4_anc']), anc_formula=xg_formula + zc_formula, ccz=ccz_code, ccz_direct=ccz_direct,
             ccz_closed_form=6 * (L - 1) ** 2)
    rows.append(r)
    ok &= r['data'] == data_formula and r['xgens'] == xg_formula and r['anc'] == r['anc_formula'] and ccz_code == ccz_direct == r['ccz_closed_form']
record('V11 qubit/generator/CCZ counts agree with direct geometric formulas (L=2..9)', ok)
pd.DataFrame(rows)

PASS V11 qubit/generator/CCZ counts agree with direct geometric formulas (L=2..9)


,L,data,data_formula,xgens,xg_formula,anc,anc_formula,ccz,ccz_direct,ccz_closed_form
0,2,28,28,11,11,27,27,6,6,6
1,3,60,60,26,26,59,59,24,24,24
2,4,104,104,47,47,103,103,54,54,54
3,5,160,160,74,74,159,159,96,96,96
4,6,228,228,107,107,227,227,150,150,150
5,7,308,308,146,146,307,307,216,216,216
6,8,400,400,191,191,399,399,294,294,294
7,9,504,504,242,242,503,503,384,384,384


## まとめ

In [37]:
n_pass = sum(v['status'] == 'PASS' for v in VAL.values()); n_fail = len(VAL) - n_pass
print(f'{n_pass} PASS, {n_fail} FAIL')
json.dump(dict(notebook='D4_switching_validation', results=VAL,
               not_verified=['Knill-Laflamme distance of the D4 phase for L >= 6 (L >= 7 with FULL_KL=True)',
                             'full code-state simulation of the protocol for L >= 3 (V7b checks the circuit on configuration inputs, V8 proves the phase formula locally)',
                             'fault tolerance / JIT decoder performance (noise not simulated)',
                             'constants of the O(d) round counts',
                             'FSC -> standard rotated-patch conversion']),
          open(f'{RESULTS}/validation_d4.json', 'w'), indent=1)
pd.DataFrame([(k, v['status']) for k, v in VAL.items()], columns=['check', 'status'])

70 PASS, 0 FAIL


,check,status
0,V1a boundary X-generator types match Fig. 4,PASS
1,"V1b boundary Z checks (Z^r on L_r, Z^b on L_b,...",PASS
2,"V1c bulk CZ positions (S^r: upper-right x2, S^...",PASS
3,"V1d CZ colour pairs (S^r: g-b, S^g: r-b, S^b: ...",PASS
4,"V2 D4 L=1: shifts in C_Z, commuting on C_Z, S^...",PASS
...,...,...
65,V10 L=3: all gates of U = W V mutually commute...,PASS
66,V10 CCZ = 7-T decomposition (matrix),PASS
67,V10 CS = 3-T decomposition (matrix),PASS
68,V10 CS^dag decomposition (matrix),PASS


### 検証できていない事項（正しさが確認されたものとして扱わない）
* $L\ge 6$ の D4 相の Knill–Laflamme 距離（$L\le5$ は V5b で厳密に $=L$；`FULL_KL=True` で $L=6$ も確認可能、約7分）。
* $L\ge3$ での符号状態全体の厳密シミュレーション（代わりに V7b：回路と位相公式の照合、V8：位相公式の局所厳密証明）。
* 耐故障性（JIT decoder の性能、しきい値）：ノイズを入れていないため未検証。
* $O(d)$ ラウンドの定数。
* FSC → 通常の回転表面符号パッチへの変換。